In [453]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [454]:
pd.set_option('display.max_columns', None)

In [455]:
df = pd.read_csv("../dataset/Restaurant.csv")
df.head()

,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,VIOLATION CODE,VIOLATION DESCRIPTION,CRITICAL FLAG,SCORE,GRADE,GRADE DATE,RECORD DATE,INSPECTION TYPE,Latitude,Longitude,Community Board,Council District,Census Tract,BIN,BBL,NTA,Location
0,50163224,BAKED CHEESE HAUS LTD,Manhattan,1,HERALD SQUARE,10001.0,6086301355,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,03/03/2026,NaN,40.750542,-73.987541,105.0,4.0,10900.0,1089436.0,1.008100e+09,MN17,POINT (-73.987541021756 40.750541907818)
1,50170678,D'S GRAB N GO,Brooklyn,74,RICHARDSON STREET,11211.0,6463025617,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,03/03/2026,NaN,40.718374,-73.949138,301.0,34.0,51500.0,3068058.0,3.027320e+09,BK73,POINT (-73.94913846043 40.718373646442)
2,50171263,NAYA ROCK CENTER LLC,Manhattan,30,ROCKEFELLER PLAZA,10112.0,2124612812,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,03/03/2026,NaN,40.758747,-73.978692,105.0,4.0,10400.0,1076262.0,1.012658e+09,MN17,POINT (-73.978692223615 40.758747437799)
3,50172771,AZ&G INC,Queens,133-33,39 AVENUE,11354.0,7188198818,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,03/03/2026,NaN,40.759110,-73.834161,407.0,20.0,87100.0,4000000.0,4.049728e+09,QN22,POINT (-73.834160749112 40.759109919719)
4,50165402,AREFIN'S TEA MANIA NEW YORK,Queens,98-04,101 AVENUE,11416.0,3478844480,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,03/03/2026,NaN,40.685049,-73.843194,409.0,32.0,4001.0,4439922.0,4.091050e+09,QN53,POINT (-73.843193921 40.685049473493)


In [456]:
df.describe()

,CAMIS,ZIPCODE,SCORE,Latitude,Longitude,Community Board,Council District,Census Tract,BIN,BBL
count,2.967660e+05,293576.000000,280188.000000,295339.000000,295339.000000,292176.000000,292203.000000,292203.000000,2.909300e+05,2.953390e+05
mean,4.806145e+07,10706.923161,25.161749,40.292099,-73.148456,254.923769,20.568711,29882.587995,2.586012e+06,2.478446e+09
std,3.755423e+06,616.227171,18.731612,4.192801,7.611227,130.374503,15.668773,31292.127581,1.356210e+06,1.338448e+09
min,3.007544e+07,6605.000000,0.000000,0.000000,-74.249101,101.000000,1.000000,100.000000,1.000000e+06,1.000000e+00
25%,5.000430e+07,10023.000000,12.000000,40.686661,-73.988620,106.000000,4.000000,8000.000000,1.051461e+06,1.011110e+09
50%,5.009207e+07,11101.000000,21.000000,40.732855,-73.955697,302.000000,20.000000,17300.000000,3.021633e+06,3.008020e+09
75%,5.012999e+07,11232.000000,33.000000,40.761237,-73.893827,402.000000,34.000000,42600.000000,4.011502e+06,4.006290e+09
max,5.018247e+07,69361.000000,203.000000,40.912822,0.000000,595.000000,51.000000,162100.000000,5.799501e+06,5.270001e+09


In [457]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 296766 entries, 0 to 296765
Data columns (total 27 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   CAMIS                  296766 non-null  int64  
 1   DBA                    296764 non-null  object 
 2   BORO                   296766 non-null  object 
 3   BUILDING               295807 non-null  object 
 4   STREET                 296765 non-null  object 
 5   ZIPCODE                293576 non-null  float64
 6   PHONE                  296752 non-null  object 
 7   CUISINE DESCRIPTION    293485 non-null  object 
 8   INSPECTION DATE        296766 non-null  object 
 9   ACTION                 293485 non-null  object 
 10  VIOLATION CODE         291191 non-null  object 
 11  VIOLATION DESCRIPTION  291190 non-null  object 
 12  CRITICAL FLAG          296766 non-null  object 
 13  SCORE                  280188 non-null  float64
 14  GRADE                  146385 non-nu

# Early Preprocessing #

## 1. Ubah ke Inspection Level ##

In [458]:
for col in ["INSPECTION DATE", "GRADE DATE", "RECORD DATE"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

record_date = df["RECORD DATE"].dropna().max()
start_date = record_date - pd.DateOffset(years=3)

df_recent = df[
    (df["INSPECTION DATE"].notna()) &
    (df["INSPECTION DATE"] >= start_date) &
    (df["INSPECTION DATE"] <= record_date)
].copy()

df_recent["SCORE"] = pd.to_numeric(df_recent["SCORE"], errors="coerce")
df_recent["BORO"] = df_recent["BORO"].replace("0", "Unknown")
df_recent["VIOLATION CODE"] = df_recent["VIOLATION CODE"].astype("string").str.strip()
df_recent["CRITICAL FLAG"] = df_recent["CRITICAL FLAG"].astype("string").str.strip().str.lower()

inspection_key = ["CAMIS", "INSPECTION DATE", "INSPECTION TYPE"]

df_ins = (
    df_recent.groupby(inspection_key, as_index=False, dropna=False)
    .agg(
        DBA=("DBA", "first"),
        BORO=("BORO", "first"),
        BUILDING=("BUILDING", "first"),
        STREET=("STREET", "first"),
        ZIPCODE=("ZIPCODE", "first"),
        PHONE=("PHONE", "first"),
        CUISINE_DESCRIPTION=("CUISINE DESCRIPTION", "first"),
        ACTION=("ACTION", "first"),
        SCORE=("SCORE", "first"),
        GRADE=("GRADE", "first"),
        GRADE_DATE=("GRADE DATE", "first"),
        Latitude=("Latitude", "first"),
        Longitude=("Longitude", "first"),
        Community_Board=("Community Board", "first"),
        Council_District=("Council District", "first"),
        Census_Tract=("Census Tract", "first"),
        BIN=("BIN", "first"),
        BBL=("BBL", "first"),
        NTA=("NTA", "first"),
        violation_count=("VIOLATION CODE", lambda x: x.dropna().ne("").sum()),
        critical_count=("CRITICAL FLAG", lambda x: (x == "critical").sum()),
        has_critical=("CRITICAL FLAG", lambda x: (x == "critical").any())
    )
)

df_ins.head()

,CAMIS,INSPECTION DATE,INSPECTION TYPE,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE_DESCRIPTION,ACTION,SCORE,GRADE,GRADE_DATE,Latitude,Longitude,Community_Board,Council_District,Census_Tract,BIN,BBL,NTA,violation_count,critical_count,has_critical
0,30075445,2023-08-01,Cycle Inspection / Initial Inspection,MORRIS PARK BAKE SHOP,Bronx,1007,MORRIS PARK AVENUE,10462.0,7188924968,Bakery Products/Desserts,Violations were cited in the following area(s).,38.0,None,NaT,40.848231,-73.855972,211.0,13.0,25200.0,2045445.0,2.041270e+09,BX37,3,2,True
1,30075445,2023-08-22,Cycle Inspection / Re-inspection,MORRIS PARK BAKE SHOP,Bronx,1007,MORRIS PARK AVENUE,10462.0,7188924968,Bakery Products/Desserts,Violations were cited in the following area(s).,12.0,A,2023-08-22,40.848231,-73.855972,211.0,13.0,25200.0,2045445.0,2.041270e+09,BX37,3,1,True
2,30075445,2024-11-08,Cycle Inspection / Initial Inspection,MORRIS PARK BAKE SHOP,Bronx,1007,MORRIS PARK AVENUE,10462.0,7188924968,Bakery Products/Desserts,Violations were cited in the following area(s).,10.0,A,2024-11-08,40.848231,-73.855972,211.0,13.0,25200.0,2045445.0,2.041270e+09,BX37,3,1,True
3,30075445,2026-02-27,Cycle Inspection / Initial Inspection,MORRIS PARK BAKE SHOP,Bronx,1007,MORRIS PARK AVENUE,10462.0,7188924968,Bakery Products/Desserts,Violations were cited in the following area(s).,7.0,A,2026-02-27,40.848231,-73.855972,211.0,13.0,25200.0,2045445.0,2.041270e+09,BX37,2,1,True
4,30191841,2023-04-23,Cycle Inspection / Initial Inspection,D.J. REYNOLDS,Manhattan,351,WEST 57 STREET,10019.0,2122452912,Irish,Violations were cited in the following area(s).,10.0,A,2023-04-23,40.767326,-73.984310,104.0,3.0,13900.0,1026048.0,1.010480e+09,MN15,2,2,True


In [459]:
df_ins.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77629 entries, 0 to 77628
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   CAMIS                77629 non-null  int64         
 1   INSPECTION DATE      77629 non-null  datetime64[ns]
 2   INSPECTION TYPE      77629 non-null  object        
 3   DBA                  77629 non-null  object        
 4   BORO                 77629 non-null  object        
 5   BUILDING             77390 non-null  object        
 6   STREET               77629 non-null  object        
 7   ZIPCODE              76772 non-null  float64       
 8   PHONE                77626 non-null  object        
 9   CUISINE_DESCRIPTION  77629 non-null  object        
 10  ACTION               77629 non-null  object        
 11  SCORE                68860 non-null  float64       
 12  GRADE                46347 non-null  object        
 13  GRADE_DATE           44221 non-

In [460]:
df_ins.duplicated().sum()

np.int64(0)

In [461]:
df_ins.isna().sum()

CAMIS                      0
INSPECTION DATE            0
INSPECTION TYPE            0
DBA                        0
BORO                       0
BUILDING                 239
STREET                     0
ZIPCODE                  857
PHONE                      3
CUISINE_DESCRIPTION        0
ACTION                     0
SCORE                   8769
GRADE                  31282
GRADE_DATE             33408
Latitude                 386
Longitude                386
Community_Board         1234
Council_District        1230
Census_Tract            1230
BIN                     1604
BBL                      386
NTA                     1234
violation_count            0
critical_count             0
has_critical               0
dtype: int64

**Drop Missing Value in Score**

In [462]:
df_ins = df_ins.dropna(subset=["SCORE"]).copy()
df_ins['SCORE'].isna().sum()

np.int64(0)

**Penjelasan:** Drop baris yang SCORE-nya null karena SCORE adalah kolom target, tidak bisa diimputasi

In [463]:
df_ins.columns

Index(['CAMIS', 'INSPECTION DATE', 'INSPECTION TYPE', 'DBA', 'BORO',
       'BUILDING', 'STREET', 'ZIPCODE', 'PHONE', 'CUISINE_DESCRIPTION',
       'ACTION', 'SCORE', 'GRADE', 'GRADE_DATE', 'Latitude', 'Longitude',
       'Community_Board', 'Council_District', 'Census_Tract', 'BIN', 'BBL',
       'NTA', 'violation_count', 'critical_count', 'has_critical'],
      dtype='object')

## 2. Ubah Tipe Data ##

In [464]:
for col in ["INSPECTION DATE", "GRADE_DATE"]:
    df_ins[col] = pd.to_datetime(df_ins[col], errors="coerce")

for col in ["SCORE", "Latitude", "Longitude"]:
    df_ins[col] = pd.to_numeric(df_ins[col], errors="coerce")

df_ins["ZIPCODE"] = df_ins["ZIPCODE"].astype("string").astype("category")

for col in ["BORO", "CUISINE_DESCRIPTION", "INSPECTION TYPE"]:
    df_ins[col] = df_ins[col].astype("category")
    
df_ins.dtypes

CAMIS                           int64
INSPECTION DATE        datetime64[ns]
INSPECTION TYPE              category
DBA                            object
BORO                         category
BUILDING                       object
STREET                         object
ZIPCODE                      category
PHONE                          object
CUISINE_DESCRIPTION          category
ACTION                         object
SCORE                         float64
GRADE                          object
GRADE_DATE             datetime64[ns]
Latitude                      float64
Longitude                     float64
Community_Board               float64
Council_District              float64
Census_Tract                  float64
BIN                           float64
BBL                           float64
NTA                            object
violation_count                 int64
critical_count                  int64
has_critical                     bool
dtype: object

**Penjelasan:** Yang Date itu krn tanggal jd pake datetime, yg score, latitude dan longitude itu bakal diproses secara numerik, selainnya itu krn emang kategorikal jadi diubah ke category aja

## 3. Fitur inti ##

In [465]:
df_ins["inspection_year"] = df_ins["INSPECTION DATE"].dt.year
df_ins["inspection_month"] = df_ins["INSPECTION DATE"].dt.month
df_ins["inspection_quarter"] = df_ins["INSPECTION DATE"].dt.quarter
df_ins["inspection_dayofweek"] = df_ins["INSPECTION DATE"].dt.dayofweek

base_features = [
    "CUISINE_DESCRIPTION",
    "BORO",
    "ZIPCODE",
    "Latitude",
    "Longitude",
    "Community_Board",
    "INSPECTION TYPE",
    "inspection_year",
    "inspection_month",
    "inspection_quarter",
    "inspection_dayofweek"
]

df_base = df_ins[
    ["CAMIS", "INSPECTION DATE", "SCORE"] + base_features
].copy()

df_base.head()

,CAMIS,INSPECTION DATE,SCORE,CUISINE_DESCRIPTION,BORO,ZIPCODE,Latitude,Longitude,Community_Board,INSPECTION TYPE,inspection_year,inspection_month,inspection_quarter,inspection_dayofweek
0,30075445,2023-08-01,38.0,Bakery Products/Desserts,Bronx,10462.0,40.848231,-73.855972,211.0,Cycle Inspection / Initial Inspection,2023,8,3,1
1,30075445,2023-08-22,12.0,Bakery Products/Desserts,Bronx,10462.0,40.848231,-73.855972,211.0,Cycle Inspection / Re-inspection,2023,8,3,1
2,30075445,2024-11-08,10.0,Bakery Products/Desserts,Bronx,10462.0,40.848231,-73.855972,211.0,Cycle Inspection / Initial Inspection,2024,11,4,4
3,30075445,2026-02-27,7.0,Bakery Products/Desserts,Bronx,10462.0,40.848231,-73.855972,211.0,Cycle Inspection / Initial Inspection,2026,2,1,4
4,30191841,2023-04-23,10.0,Irish,Manhattan,10019.0,40.767326,-73.984310,104.0,Cycle Inspection / Initial Inspection,2023,4,2,6


In [466]:
df_base.columns

Index(['CAMIS', 'INSPECTION DATE', 'SCORE', 'CUISINE_DESCRIPTION', 'BORO',
       'ZIPCODE', 'Latitude', 'Longitude', 'Community_Board',
       'INSPECTION TYPE', 'inspection_year', 'inspection_month',
       'inspection_quarter', 'inspection_dayofweek'],
      dtype='object')

In [467]:
df_base.dtypes

CAMIS                            int64
INSPECTION DATE         datetime64[ns]
SCORE                          float64
CUISINE_DESCRIPTION           category
BORO                          category
ZIPCODE                       category
Latitude                       float64
Longitude                      float64
Community_Board                float64
INSPECTION TYPE               category
inspection_year                  int32
inspection_month                 int32
inspection_quarter               int32
inspection_dayofweek             int32
dtype: object

## 4. Handle Missing Value Boro, Zipcode, Cuisine Description, Inspection Type ##

In [468]:
cat_cols = ["BORO", "ZIPCODE", "CUISINE_DESCRIPTION", "INSPECTION TYPE"]

for col in cat_cols:
    df_base[col] = df_base[col].replace(r"^\s*$", pd.NA, regex=True)

    if str(df_base[col].dtype) == "category":
        if "Unknown" not in df_base[col].cat.categories:
            df_base[col] = df_base[col].cat.add_categories(["Unknown"])

    df_base[col] = df_base[col].fillna("Unknown")
    
df_base[cat_cols].isna().sum()

BORO                   0
ZIPCODE                0
CUISINE_DESCRIPTION    0
INSPECTION TYPE        0
dtype: int64

In [469]:
df_base.info()

<class 'pandas.core.frame.DataFrame'>
Index: 68860 entries, 0 to 77627
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   CAMIS                 68860 non-null  int64         
 1   INSPECTION DATE       68860 non-null  datetime64[ns]
 2   SCORE                 68860 non-null  float64       
 3   CUISINE_DESCRIPTION   68860 non-null  category      
 4   BORO                  68860 non-null  category      
 5   ZIPCODE               68860 non-null  category      
 6   Latitude              68505 non-null  float64       
 7   Longitude             68505 non-null  float64       
 8   Community_Board       67732 non-null  float64       
 9   INSPECTION TYPE       68860 non-null  category      
 10  inspection_year       68860 non-null  int32         
 11  inspection_month      68860 non-null  int32         
 12  inspection_quarter    68860 non-null  int32         
 13  inspection_dayofweek 

## 5. Handle Missing Value Community Board ##

In [470]:
df_base["Community_Board"] = df_base["Community_Board"].astype("string")
df_base["Community_Board"] = df_base["Community_Board"].replace(r"^\s*$", pd.NA, regex=True)
df_base["Community_Board"] = df_base["Community_Board"].fillna("Unknown")

## 6. Handle Missing Value Latitude Longitude ##

In [471]:
df_base[['Latitude', 'Longitude']].isna().sum()

Latitude     355
Longitude    355
dtype: int64

In [472]:
df_base["Latitude"] = pd.to_numeric(df_base["Latitude"], errors="coerce")
df_base["Longitude"] = pd.to_numeric(df_base["Longitude"], errors="coerce")

df_base.loc[df_base["Latitude"] == 0, "Latitude"] = pd.NA
df_base.loc[df_base["Longitude"] == 0, "Longitude"] = pd.NA

df_base["Latitude"] = df_base["Latitude"].fillna(
    df_base.groupby("CAMIS")["Latitude"].transform("median")
)

df_base["Longitude"] = df_base["Longitude"].fillna(
    df_base.groupby("CAMIS")["Longitude"].transform("median")
)

df_base["Latitude"] = df_base["Latitude"].fillna(df_base["Latitude"].median())
df_base["Longitude"] = df_base["Longitude"].fillna(df_base["Longitude"].median())

In [473]:
df_base[['Latitude', 'Longitude']].isna().sum()

Latitude     0
Longitude    0
dtype: int64

In [474]:
df_base.isna().sum()

CAMIS                   0
INSPECTION DATE         0
SCORE                   0
CUISINE_DESCRIPTION     0
BORO                    0
ZIPCODE                 0
Latitude                0
Longitude               0
Community_Board         0
INSPECTION TYPE         0
inspection_year         0
inspection_month        0
inspection_quarter      0
inspection_dayofweek    0
dtype: int64

In [475]:
df_base.info()

<class 'pandas.core.frame.DataFrame'>
Index: 68860 entries, 0 to 77627
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   CAMIS                 68860 non-null  int64         
 1   INSPECTION DATE       68860 non-null  datetime64[ns]
 2   SCORE                 68860 non-null  float64       
 3   CUISINE_DESCRIPTION   68860 non-null  category      
 4   BORO                  68860 non-null  category      
 5   ZIPCODE               68860 non-null  category      
 6   Latitude              68860 non-null  float64       
 7   Longitude             68860 non-null  float64       
 8   Community_Board       68860 non-null  string        
 9   INSPECTION TYPE       68860 non-null  category      
 10  inspection_year       68860 non-null  int32         
 11  inspection_month      68860 non-null  int32         
 12  inspection_quarter    68860 non-null  int32         
 13  inspection_dayofweek 

# Advanced Preprocessing #

In [476]:
df_adv = df_base.copy()

extra_cols = [
    "CAMIS",
    "INSPECTION DATE",
    "GRADE",
    "violation_count",
    "critical_count",
    "DBA",
    "ACTION"
]

df_extra = df_ins[extra_cols].copy()

df_adv = df_adv.merge(
    df_extra,
    on=["CAMIS", "INSPECTION DATE"],
    how="left"
)

print("Shape df_adv:", df_adv.shape)
print(df_adv.columns.tolist())

Shape df_adv: (68860, 19)
['CAMIS', 'INSPECTION DATE', 'SCORE', 'CUISINE_DESCRIPTION', 'BORO', 'ZIPCODE', 'Latitude', 'Longitude', 'Community_Board', 'INSPECTION TYPE', 'inspection_year', 'inspection_month', 'inspection_quarter', 'inspection_dayofweek', 'GRADE', 'violation_count', 'critical_count', 'DBA', 'ACTION']


**Penjelasan:** Pada tahap advanced preprocessing, dataframe kerja dibuat dari `df_base` karena `df_base` merupakan hasil akhir dari early preprocessing yang sudah berisi fitur-fitur dasar yang telah dibersihkan. Namun, karena advanced preprocessing membutuhkan informasi tambahan seperti `GRADE`, `GRADE_DATE`, `violation_count`, dan `critical_count` untuk membentuk fitur historis, maka beberapa kolom tersebut ditambahkan kembali dari `df_ins` ke dalam dataframe baru bernama `df_adv`. Dengan demikian, pipeline tetap konsisten dengan hasil early preprocessing, tetapi masih memiliki informasi yang cukup untuk feature engineering lanjutan.

## 1. Sorting per restoran dan pembuatan urutan inspeksi ##
Pada tahap ini, data diurutkan per restoran berdasarkan waktu inspeksi agar setiap baris memiliki posisi kronologis yang jelas. Urutan ini akan menjadi dasar untuk membangun fitur historis seperti `prev_score_1`, `prev_grade`, `days_since_last_inspection`, dan fitur riwayat lainnya pada tahap berikutnya.Sort Data berdasarkan Inspection Date

In [477]:
df_adv = df_adv.sort_values(
    by=["CAMIS", "INSPECTION DATE", "INSPECTION TYPE"],
    ascending=[True, True, True],
    kind="mergesort"  
).reset_index(drop=True)

df_adv["inspection_seq"] = df_adv.groupby("CAMIS").cumcount() + 1

df_adv["is_first_inspection"] = (df_adv["inspection_seq"] == 1).astype(int)
df_adv["has_history"] = (df_adv["inspection_seq"] > 1).astype(int)

df_adv[["CAMIS", "INSPECTION DATE", "INSPECTION TYPE", "inspection_seq", "is_first_inspection", "has_history"]].head(15)

,CAMIS,INSPECTION DATE,INSPECTION TYPE,inspection_seq,is_first_inspection,has_history
0,30075445,2023-08-01,Cycle Inspection / Initial Inspection,1,1,0
1,30075445,2023-08-22,Cycle Inspection / Re-inspection,2,0,1
2,30075445,2024-11-08,Cycle Inspection / Initial Inspection,3,0,1
3,30075445,2026-02-27,Cycle Inspection / Initial Inspection,4,0,1
4,30191841,2023-04-23,Cycle Inspection / Initial Inspection,1,1,0
5,30191841,2024-11-20,Cycle Inspection / Initial Inspection,2,0,1
6,30191841,2025-02-20,Cycle Inspection / Re-inspection,3,0,1
7,40356018,2024-04-16,Cycle Inspection / Initial Inspection,1,1,0
8,40356018,2025-09-17,Cycle Inspection / Initial Inspection,2,0,1
9,40356483,2023-11-16,Cycle Inspection / Initial Inspection,1,1,0


In [478]:
sample_camis = df_adv["CAMIS"].drop_duplicates().head(5)

display(
    df_adv[df_adv["CAMIS"].isin(sample_camis)]
    [["CAMIS", "DBA", "INSPECTION DATE", "INSPECTION TYPE", "SCORE", "inspection_seq"]]
    .sort_values(["CAMIS", "INSPECTION DATE", "INSPECTION TYPE"])
)

print("Jumlah restoran unik:", df_adv["CAMIS"].nunique())
print("Maks urutan inspeksi:", df_adv["inspection_seq"].max())
print("Proporsi baris yang punya history:", df_adv["has_history"].mean())

,CAMIS,DBA,INSPECTION DATE,INSPECTION TYPE,SCORE,inspection_seq
0,30075445,MORRIS PARK BAKE SHOP,2023-08-01,Cycle Inspection / Initial Inspection,38.0,1
1,30075445,MORRIS PARK BAKE SHOP,2023-08-22,Cycle Inspection / Re-inspection,12.0,2
2,30075445,MORRIS PARK BAKE SHOP,2024-11-08,Cycle Inspection / Initial Inspection,10.0,3
3,30075445,MORRIS PARK BAKE SHOP,2026-02-27,Cycle Inspection / Initial Inspection,7.0,4
4,30191841,D.J. REYNOLDS,2023-04-23,Cycle Inspection / Initial Inspection,10.0,1
5,30191841,D.J. REYNOLDS,2024-11-20,Cycle Inspection / Initial Inspection,24.0,2
6,30191841,D.J. REYNOLDS,2025-02-20,Cycle Inspection / Re-inspection,10.0,3
7,40356018,RIVIERA CATERERS,2024-04-16,Cycle Inspection / Initial Inspection,0.0,1
8,40356018,RIVIERA CATERERS,2025-09-17,Cycle Inspection / Initial Inspection,10.0,2
9,40356483,WILKEN'S FINE FOOD,2023-11-16,Cycle Inspection / Initial Inspection,35.0,1


Jumlah restoran unik: 27071
Maks urutan inspeksi: 14
Proporsi baris yang punya history: 0.6068690095846645


**Penjelasan:** Data diurutkan berdasarkan `CAMIS`, `INSPECTION DATE`, dan `INSPECTION TYPE` untuk memastikan setiap inspeksi berada dalam urutan kronologis yang benar di dalam masing-masing restoran. Setelah itu dibuat kolom `inspection_seq` sebagai nomor urut inspeksi per restoran, sehingga setiap baris dapat diketahui apakah merupakan inspeksi pertama atau sudah memiliki riwayat. Langkah ini penting karena seluruh fitur historis pada tahap berikutnya harus dibangun hanya dari inspeksi sebelumnya agar tidak terjadi leakage.

In [479]:
df_adv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68860 entries, 0 to 68859
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   CAMIS                 68860 non-null  int64         
 1   INSPECTION DATE       68860 non-null  datetime64[ns]
 2   SCORE                 68860 non-null  float64       
 3   CUISINE_DESCRIPTION   68860 non-null  category      
 4   BORO                  68860 non-null  category      
 5   ZIPCODE               68860 non-null  category      
 6   Latitude              68860 non-null  float64       
 7   Longitude             68860 non-null  float64       
 8   Community_Board       68860 non-null  string        
 9   INSPECTION TYPE       68860 non-null  category      
 10  inspection_year       68860 non-null  int32         
 11  inspection_month      68860 non-null  int32         
 12  inspection_quarter    68860 non-null  int32         
 13  inspection_dayof

## 2. Historical Features ##

Pada tahap ini dibuat fitur-fitur riwayat per restoran berdasarkan urutan inspeksi yang sudah dibentuk sebelumnya. Seluruh fitur historis dihitung hanya dari inspeksi sebelumnya menggunakan mekanisme `shift`, sehingga informasi dari inspeksi saat ini tidak ikut digunakan dan risiko leakage dapat dihindari. Fitur yang dibangun mencakup riwayat score, riwayat pelanggaran, riwayat waktu inspeksi, serta riwayat outcome dari grade sebelumnya.

In [480]:
df_hist = df_adv.copy()

if "GRADE" in df_hist.columns:
    df_hist["GRADE"] = (
        df_hist["GRADE"]
        .astype("string")
        .str.strip()
    )

g = df_hist.groupby("CAMIS", sort=False)

df_hist["prev_score_1"] = g["SCORE"].shift(1)
df_hist["prev_score_2"] = g["SCORE"].shift(2)

df_hist["avg_prev_score"] = g["SCORE"].transform(
    lambda s: s.shift(1).expanding().mean()
)

df_hist["max_prev_score"] = g["SCORE"].transform(
    lambda s: s.shift(1).expanding().max()
)

df_hist["score_trend"] = df_hist["prev_score_1"] - df_hist["prev_score_2"]

df_hist["score_delta_last"] = df_hist["prev_score_1"] - df_hist["avg_prev_score"]


df_hist["prev_violation_count"] = g["violation_count"].shift(1)
df_hist["prev_critical_count"] = g["critical_count"].shift(1)

df_hist["avg_prev_critical_count"] = g["critical_count"].transform(
    lambda s: s.shift(1).expanding().mean()
)

df_hist["prior_critical_inspection_count"] = g["critical_count"].transform(
    lambda s: s.shift(1).gt(0).cumsum()
)

df_hist["repeat_offender_flag"] = (
    df_hist["prior_critical_inspection_count"] >= 2
).astype("int8")


df_hist["prev_inspection_date"] = g["INSPECTION DATE"].shift(1)

df_hist["days_since_last_inspection"] = (
    df_hist["INSPECTION DATE"] - df_hist["prev_inspection_date"]
).dt.days

df_hist["first_inspection_date"] = g["INSPECTION DATE"].transform("min")

df_hist["days_since_first_inspection"] = (
    df_hist["INSPECTION DATE"] - df_hist["first_inspection_date"]
).dt.days

df_hist["inspection_count_so_far"] = df_hist["inspection_seq"] - 1


df_hist["prev_grade"] = g["GRADE"].shift(1)

df_hist["ever_failed_gradeA_flag"] = g["GRADE"].transform(
    lambda s: (s.shift(1).notna() & s.shift(1).ne("A")).cummax()
).astype("int8")

risk_map = {
    "A": "low_risk",
    "B": "medium_risk",
    "C": "high_risk"
}

df_hist["last_inspection_risk_bucket"] = (
    df_hist["prev_grade"].map(risk_map).fillna("Unknown")
)

print("Shape df_hist:", df_hist.shape)

Shape df_hist: (68860, 41)


In [481]:
df_hist.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68860 entries, 0 to 68859
Data columns (total 41 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   CAMIS                            68860 non-null  int64         
 1   INSPECTION DATE                  68860 non-null  datetime64[ns]
 2   SCORE                            68860 non-null  float64       
 3   CUISINE_DESCRIPTION              68860 non-null  category      
 4   BORO                             68860 non-null  category      
 5   ZIPCODE                          68860 non-null  category      
 6   Latitude                         68860 non-null  float64       
 7   Longitude                        68860 non-null  float64       
 8   Community_Board                  68860 non-null  string        
 9   INSPECTION TYPE                  68860 non-null  category      
 10  inspection_year                  68860 non-null  int32    

In [482]:
df_hist['repeat_offender_flag'].value_counts()

repeat_offender_flag
0    51070
1    17790
Name: count, dtype: int64

In [483]:
df_hist.head(5)

,CAMIS,INSPECTION DATE,SCORE,CUISINE_DESCRIPTION,BORO,ZIPCODE,Latitude,Longitude,Community_Board,INSPECTION TYPE,inspection_year,inspection_month,inspection_quarter,inspection_dayofweek,GRADE,violation_count,critical_count,DBA,ACTION,inspection_seq,is_first_inspection,has_history,prev_score_1,prev_score_2,avg_prev_score,max_prev_score,score_trend,score_delta_last,prev_violation_count,prev_critical_count,avg_prev_critical_count,prior_critical_inspection_count,repeat_offender_flag,prev_inspection_date,days_since_last_inspection,first_inspection_date,days_since_first_inspection,inspection_count_so_far,prev_grade,ever_failed_gradeA_flag,last_inspection_risk_bucket
0,30075445,2023-08-01,38.0,Bakery Products/Desserts,Bronx,10462.0,40.848231,-73.855972,211.0,Cycle Inspection / Initial Inspection,2023,8,3,1,<NA>,3,2,MORRIS PARK BAKE SHOP,Violations were cited in the following area(s).,1,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaT,NaN,2023-08-01,0,0,<NA>,0,Unknown
1,30075445,2023-08-22,12.0,Bakery Products/Desserts,Bronx,10462.0,40.848231,-73.855972,211.0,Cycle Inspection / Re-inspection,2023,8,3,1,A,3,1,MORRIS PARK BAKE SHOP,Violations were cited in the following area(s).,2,0,1,38.0,NaN,38.0,38.0,NaN,0.0,3.0,2.0,2.000000,1,0,2023-08-01,21.0,2023-08-01,21,1,<NA>,0,Unknown
2,30075445,2024-11-08,10.0,Bakery Products/Desserts,Bronx,10462.0,40.848231,-73.855972,211.0,Cycle Inspection / Initial Inspection,2024,11,4,4,A,3,1,MORRIS PARK BAKE SHOP,Violations were cited in the following area(s).,3,0,1,12.0,38.0,25.0,38.0,-26.0,-13.0,3.0,1.0,1.500000,2,1,2023-08-22,444.0,2023-08-01,465,2,A,0,low_risk
3,30075445,2026-02-27,7.0,Bakery Products/Desserts,Bronx,10462.0,40.848231,-73.855972,211.0,Cycle Inspection / Initial Inspection,2026,2,1,4,A,2,1,MORRIS PARK BAKE SHOP,Violations were cited in the following area(s).,4,0,1,10.0,12.0,20.0,38.0,-2.0,-10.0,3.0,1.0,1.333333,3,1,2024-11-08,476.0,2023-08-01,941,3,A,0,low_risk
4,30191841,2023-04-23,10.0,Irish,Manhattan,10019.0,40.767326,-73.984310,104.0,Cycle Inspection / Initial Inspection,2023,4,2,6,A,2,2,D.J. REYNOLDS,Violations were cited in the following area(s).,1,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaT,NaN,2023-04-23,0,0,<NA>,0,Unknown


**Penjelasan:** Fitur historis dibangun untuk menangkap pola kepatuhan restoran dari inspeksi-inspeksi sebelumnya. Seluruh fitur dibuat secara kronologis per `CAMIS` menggunakan informasi masa lalu saja, seperti skor sebelumnya, jumlah pelanggaran sebelumnya, jarak waktu antar inspeksi, dan grade sebelumnya, sehingga tidak terjadi leakage dari inspeksi saat ini. Missing value yang muncul pada beberapa fitur historis, terutama pada inspeksi pertama atau kedua, dianggap wajar karena restoran tersebut memang belum memiliki riwayat yang cukup pada saat itu.

## 3. Missing Value Handling Setelah Historical Features

Setelah fitur historis dibentuk, beberapa missing value muncul secara alami karena tidak semua restoran memiliki riwayat inspeksi yang cukup. Missing pada fitur seperti `prev_score_1`, `prev_score_2`, `prev_grade`, dan `days_since_last_inspection` dianggap sebagai missing struktural, bukan error data. Oleh karena itu, penanganannya dilakukan dengan pendekatan yang sesuai konteks: fitur count diisi 0, fitur kategorikal diisi `Unknown`, dan fitur numerik historis diisi median agar distribusi data tetap stabil.

In [484]:
df_hist_clean = df_hist.copy()

categorical_hist_cols = [
    "prev_grade",
    "last_inspection_risk_bucket"
]

for col in categorical_hist_cols:
    if col in df_hist_clean.columns:
        df_hist_clean[col] = df_hist_clean[col].fillna("Unknown")

zero_fill_cols = [
    "prev_violation_count",
    "prev_critical_count",
    "avg_prev_critical_count",
    "prior_critical_inspection_count",
    "inspection_count_so_far",
    "repeat_offender_flag",
    "ever_failed_gradeA_flag"
]

for col in zero_fill_cols:
    if col in df_hist_clean.columns:
        df_hist_clean[col] = df_hist_clean[col].fillna(0)

history_num_cols = [
    "prev_score_1",
    "prev_score_2",
    "avg_prev_score",
    "max_prev_score",
    "days_since_last_inspection"
]

for col in history_num_cols:
    if col in df_hist_clean.columns:
        df_hist_clean[f"{col}_missing_flag"] = df_hist_clean[col].isna().astype("int8")
        df_hist_clean[col] = df_hist_clean[col].fillna(df_hist_clean[col].median())

zero_or_median_cols = {
    "score_trend": 0,
    "score_delta_last": 0,
    "days_since_first_inspection": 0
}

for col, fill_value in zero_or_median_cols.items():
    if col in df_hist_clean.columns:
        df_hist_clean[col] = df_hist_clean[col].fillna(fill_value)

categorical_cols = df_hist_clean.select_dtypes(include=["object", "category", "string"]).columns.tolist()
for col in categorical_cols:
    if col not in ["GRADE"]:  
        df_hist_clean[col] = df_hist_clean[col].fillna("Unknown")

numeric_cols = df_hist_clean.select_dtypes(include=["number"]).columns.tolist()
for col in numeric_cols:
    if col != "SCORE" and df_hist_clean[col].isna().any():
        df_hist_clean[col] = df_hist_clean[col].fillna(df_hist_clean[col].median())

if "prev_inspection_date" in df_hist_clean.columns:
    df_hist_clean["prev_inspection_date_missing_flag"] = df_hist_clean["prev_inspection_date"].isna().astype("int8")
    df_hist_clean["prev_inspection_date"] = df_hist_clean["prev_inspection_date"].fillna(df_hist_clean["INSPECTION DATE"])

df_hist_clean["days_since_last_inspection"] = (
    df_hist_clean["INSPECTION DATE"] - df_hist_clean["prev_inspection_date"]
).dt.days
    
print("Shape df_hist_clean:", df_hist_clean.shape)

Shape df_hist_clean: (68860, 47)


In [485]:
df_hist_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68860 entries, 0 to 68859
Data columns (total 47 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   CAMIS                                    68860 non-null  int64         
 1   INSPECTION DATE                          68860 non-null  datetime64[ns]
 2   SCORE                                    68860 non-null  float64       
 3   CUISINE_DESCRIPTION                      68860 non-null  category      
 4   BORO                                     68860 non-null  category      
 5   ZIPCODE                                  68860 non-null  category      
 6   Latitude                                 68860 non-null  float64       
 7   Longitude                                68860 non-null  float64       
 8   Community_Board                          68860 non-null  string        
 9   INSPECTION TYPE                        

In [486]:
df_hist_clean.head()

,CAMIS,INSPECTION DATE,SCORE,CUISINE_DESCRIPTION,BORO,ZIPCODE,Latitude,Longitude,Community_Board,INSPECTION TYPE,inspection_year,inspection_month,inspection_quarter,inspection_dayofweek,GRADE,violation_count,critical_count,DBA,ACTION,inspection_seq,is_first_inspection,has_history,prev_score_1,prev_score_2,avg_prev_score,max_prev_score,score_trend,score_delta_last,prev_violation_count,prev_critical_count,avg_prev_critical_count,prior_critical_inspection_count,repeat_offender_flag,prev_inspection_date,days_since_last_inspection,first_inspection_date,days_since_first_inspection,inspection_count_so_far,prev_grade,ever_failed_gradeA_flag,last_inspection_risk_bucket,prev_score_1_missing_flag,prev_score_2_missing_flag,avg_prev_score_missing_flag,max_prev_score_missing_flag,days_since_last_inspection_missing_flag,prev_inspection_date_missing_flag
0,30075445,2023-08-01,38.0,Bakery Products/Desserts,Bronx,10462.0,40.848231,-73.855972,211.0,Cycle Inspection / Initial Inspection,2023,8,3,1,<NA>,3,2,MORRIS PARK BAKE SHOP,Violations were cited in the following area(s).,1,1,0,15.0,19.0,18.714286,23.0,0.0,0.0,0.0,0.0,0.000000,0,0,2023-08-01,0,2023-08-01,0,0,Unknown,0,Unknown,1,1,1,1,1,1
1,30075445,2023-08-22,12.0,Bakery Products/Desserts,Bronx,10462.0,40.848231,-73.855972,211.0,Cycle Inspection / Re-inspection,2023,8,3,1,A,3,1,MORRIS PARK BAKE SHOP,Violations were cited in the following area(s).,2,0,1,38.0,19.0,38.000000,38.0,0.0,0.0,3.0,2.0,2.000000,1,0,2023-08-01,21,2023-08-01,21,1,Unknown,0,Unknown,0,1,0,0,0,0
2,30075445,2024-11-08,10.0,Bakery Products/Desserts,Bronx,10462.0,40.848231,-73.855972,211.0,Cycle Inspection / Initial Inspection,2024,11,4,4,A,3,1,MORRIS PARK BAKE SHOP,Violations were cited in the following area(s).,3,0,1,12.0,38.0,25.000000,38.0,-26.0,-13.0,3.0,1.0,1.500000,2,1,2023-08-22,444,2023-08-01,465,2,A,0,low_risk,0,0,0,0,0,0
3,30075445,2026-02-27,7.0,Bakery Products/Desserts,Bronx,10462.0,40.848231,-73.855972,211.0,Cycle Inspection / Initial Inspection,2026,2,1,4,A,2,1,MORRIS PARK BAKE SHOP,Violations were cited in the following area(s).,4,0,1,10.0,12.0,20.000000,38.0,-2.0,-10.0,3.0,1.0,1.333333,3,1,2024-11-08,476,2023-08-01,941,3,A,0,low_risk,0,0,0,0,0,0
4,30191841,2023-04-23,10.0,Irish,Manhattan,10019.0,40.767326,-73.984310,104.0,Cycle Inspection / Initial Inspection,2023,4,2,6,A,2,2,D.J. REYNOLDS,Violations were cited in the following area(s).,1,1,0,15.0,19.0,18.714286,23.0,0.0,0.0,0.0,0.0,0.000000,0,0,2023-04-23,0,2023-04-23,0,0,Unknown,0,Unknown,1,1,1,1,1,1


**Penjelasan:** Setelah fitur historis dibentuk, beberapa missing value muncul secara alami karena tidak semua restoran memiliki riwayat inspeksi yang cukup. Missing pada fitur seperti `prev_score_1`, `prev_score_2`, `prev_grade`, dan `days_since_last_inspection` dianggap sebagai missing struktural, bukan error data. Oleh karena itu, penanganannya dilakukan dengan pendekatan yang sesuai konteks: fitur count diisi dengan median lalu ditambahkan kolom pembantu yang isinya adalah flag yang menentukan apakah itu merupakan inspeksi pertama atau bukan, fitur kategorikal diisi `Unknown`, dan fitur numerik historis diisi median agar distribusi data tetap stabil.

**Penjelasan:** Simple time split digunakan untuk menjaga urutan waktu antara data latih, validasi, dan pengujian. Dengan cara ini, model hanya belajar dari data inspeksi masa lalu dan dievaluasi pada periode yang lebih baru, sehingga hasil evaluasi lebih mencerminkan performa saat diterapkan di dunia nyata. Setelah split dilakukan, kolom `SCORE` dipisahkan sebagai target, sedangkan `GRADE` dan `GRADE_DATE` dibuang dari fitur karena berpotensi menimbulkan leakage.

## 4. Historical Group Features dari Cuisine, Boro, dan Inspection Type

Selain riwayat per restoran, fitur historis juga dibuat pada level kelompok untuk menangkap pola risiko yang lebih umum. Pada tahap ini dihitung rata-rata historical score per `CUISINE DESCRIPTION`, critical violation rate historis per `BORO`, dan proporsi inspeksi buruk historis per `INSPECTION TYPE`. Seluruh agregasi dibuat secara kronologis dengan hanya menggunakan data dari periode sebelumnya agar tidak terjadi leakage dari inspeksi saat ini.

In [487]:
df_group_hist = df_hist_clean.copy()

df_group_hist = df_group_hist.sort_values(
    by=["INSPECTION DATE", "CAMIS", "inspection_seq"],
    ascending=[True, True, True],
    kind="mergesort"
).reset_index(drop=True)

df_group_hist["bad_inspection_flag"] = (df_group_hist["SCORE"] >= 14).astype("int8")

g_cuisine = df_group_hist.groupby("CUISINE_DESCRIPTION", sort=False)

df_group_hist["cuisine_hist_count"] = g_cuisine.cumcount()

df_group_hist["cuisine_hist_avg_score"] = g_cuisine["SCORE"].transform(
    lambda s: s.shift(1).expanding().mean()
)

df_group_hist["cuisine_hist_max_score"] = g_cuisine["SCORE"].transform(
    lambda s: s.shift(1).expanding().max()
)

df_group_hist["critical_violation_flag"] = (
    df_group_hist["critical_count"] > 0
).astype("int8")

g_boro = df_group_hist.groupby("BORO", sort=False)

df_group_hist["boro_hist_count"] = g_boro.cumcount()

df_group_hist["boro_hist_critical_rate"] = g_boro["critical_violation_flag"].transform(
    lambda s: s.shift(1).expanding().mean()
)

g_insp = df_group_hist.groupby("INSPECTION TYPE", sort=False)

df_group_hist["insp_type_hist_count"] = g_insp.cumcount()

df_group_hist["insp_type_hist_bad_rate"] = g_insp["bad_inspection_flag"].transform(
    lambda s: s.shift(1).expanding().mean()
)

print("Shape df_group_hist:", df_group_hist.shape)

Shape df_group_hist: (68860, 56)


C:\Users\MSI SWORD 16\AppData\Local\Temp\ipykernel_34084\179633225.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g_cuisine = df_group_hist.groupby("CUISINE_DESCRIPTION", sort=False)
C:\Users\MSI SWORD 16\AppData\Local\Temp\ipykernel_34084\179633225.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g_boro = df_group_hist.groupby("BORO", sort=False)
C:\Users\MSI SWORD 16\AppData\Local\Temp\ipykernel_34084\179633225.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True 

In [488]:
group_hist_cols = [
    "INSPECTION DATE",
    "CAMIS",
    "CUISINE_DESCRIPTION",
    "BORO",
    "INSPECTION TYPE",
    "SCORE",
    "critical_count",
    "bad_inspection_flag",
    "cuisine_hist_count",
    "cuisine_hist_avg_score",
    "cuisine_hist_max_score",
    "boro_hist_count",
    "boro_hist_critical_rate",
    "insp_type_hist_count",
    "insp_type_hist_bad_rate"
]

display(df_group_hist[group_hist_cols].head(15))

,INSPECTION DATE,CAMIS,CUISINE_DESCRIPTION,BORO,INSPECTION TYPE,SCORE,critical_count,bad_inspection_flag,cuisine_hist_count,cuisine_hist_avg_score,cuisine_hist_max_score,boro_hist_count,boro_hist_critical_rate,insp_type_hist_count,insp_type_hist_bad_rate
0,2023-03-03,40385774,American,Brooklyn,Cycle Inspection / Re-inspection,45.0,5,1,0,NaN,NaN,0,NaN,0,NaN
1,2023-03-03,40397359,Irish,Manhattan,Cycle Inspection / Initial Inspection,28.0,2,1,0,NaN,NaN,0,NaN,0,NaN
2,2023-03-03,40423830,Caribbean,Bronx,Cycle Inspection / Reopening Inspection,11.0,2,0,0,NaN,NaN,0,NaN,0,NaN
3,2023-03-03,40534912,Caribbean,Brooklyn,Cycle Inspection / Initial Inspection,19.0,2,1,1,11.000000,11.0,1,1.000000,1,1.000000
4,2023-03-03,40676884,American,Brooklyn,Cycle Inspection / Re-inspection,21.0,3,1,1,45.000000,45.0,2,1.000000,1,1.000000
5,2023-03-03,40988630,American,Manhattan,Cycle Inspection / Initial Inspection,2.0,0,0,2,33.000000,45.0,1,1.000000,2,1.000000
6,2023-03-03,40992170,American,Brooklyn,Cycle Inspection / Initial Inspection,5.0,1,0,3,22.666667,45.0,3,1.000000,3,0.666667
7,2023-03-03,41236257,Donuts,Brooklyn,Cycle Inspection / Initial Inspection,49.0,5,1,0,NaN,NaN,4,1.000000,4,0.500000
8,2023-03-03,41314872,Pizza,Bronx,Cycle Inspection / Initial Inspection,10.0,1,0,0,NaN,NaN,1,1.000000,5,0.600000
9,2023-03-03,41403890,American,Brooklyn,Cycle Inspection / Reopening Inspection,0.0,0,0,4,18.250000,45.0,5,1.000000,1,0.000000


In [489]:
group_missing_summary = (
    df_group_hist[
        [
            "cuisine_hist_avg_score",
            "cuisine_hist_max_score",
            "boro_hist_critical_rate",
            "insp_type_hist_bad_rate"
        ]
    ]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_ratio")
    .to_frame()
)

display(group_missing_summary)

,missing_ratio
cuisine_hist_avg_score,0.001307
cuisine_hist_max_score,0.001307
insp_type_hist_bad_rate,0.000232
boro_hist_critical_rate,0.000087


## 5. Data Splitting dengan Simple Time Split

Data dibagi berdasarkan tahun inspeksi agar proses evaluasi mengikuti urutan waktu yang realistis. Data tahun 2023–2024 digunakan sebagai training set, tahun 2025 sebagai validation set, dan tahun 2026 sebagai test set. Pendekatan ini lebih sesuai untuk data inspeksi yang memiliki dependensi temporal dibandingkan random split biasa.

In [490]:
df_split = df_group_hist.copy()

train_df = df_split[df_split["inspection_year"].isin([2023, 2024])].copy()
val_df   = df_split[df_split["inspection_year"] == 2025].copy()
test_df  = df_split[df_split["inspection_year"] == 2026].copy()

print("Train shape:", train_df.shape)
print("Val shape  :", val_df.shape)
print("Test shape :", test_df.shape)

Train shape: (39538, 56)
Val shape  : (25590, 56)
Test shape : (3732, 56)


In [491]:
print("Distribusi tahun total:")
print(df_split["inspection_year"].value_counts().sort_index())
print()

print("Distribusi train:")
print(train_df["inspection_year"].value_counts().sort_index())
print()

print("Distribusi val:")
print(val_df["inspection_year"].value_counts().sort_index())
print()

print("Distribusi test:")
print(test_df["inspection_year"].value_counts().sort_index())

Distribusi tahun total:
inspection_year
2023    15559
2024    23979
2025    25590
2026     3732
Name: count, dtype: int64

Distribusi train:
inspection_year
2023    15559
2024    23979
Name: count, dtype: int64

Distribusi val:
inspection_year
2025    25590
Name: count, dtype: int64

Distribusi test:
inspection_year
2026    3732
Name: count, dtype: int64


In [492]:
target_col = "SCORE"

leakage_cols = [
    "SCORE",
    "GRADE"
]

drop_if_exists = [col for col in leakage_cols if col in df_split.columns]

X_train = train_df.drop(columns=[col for col in drop_if_exists if col in train_df.columns]).copy()
y_train = train_df[target_col].copy()

X_val = val_df.drop(columns=[col for col in drop_if_exists if col in val_df.columns]).copy()
y_val = val_df[target_col].copy()

X_test = test_df.drop(columns=[col for col in drop_if_exists if col in test_df.columns]).copy()
y_test = test_df[target_col].copy()

print("X_train:", X_train.shape, ", y_train:", y_train.shape)
print("X_val  :", X_val.shape,   ", y_val  :", y_val.shape)
print("X_test :", X_test.shape,  ", y_test :", y_test.shape)

X_train: (39538, 54) , y_train: (39538,)
X_val  : (25590, 54) , y_val  : (25590,)
X_test : (3732, 54) , y_test : (3732,)


In [493]:
for col in ["SCORE", "GRADE"]:
    print(
        col,
        ", train:", col in X_train.columns,
        ", val:", col in X_val.columns,
        ", test:", col in X_test.columns
    )

SCORE , train: False , val: False , test: False
GRADE , train: False , val: False , test: False


In [494]:
print("Rentang tanggal train :", train_df["INSPECTION DATE"].min(), "s.d.", train_df["INSPECTION DATE"].max())
print("Rentang tanggal val   :", val_df["INSPECTION DATE"].min(), "s.d.", val_df["INSPECTION DATE"].max())
print("Rentang tanggal test  :", test_df["INSPECTION DATE"].min(), "s.d.", test_df["INSPECTION DATE"].max())

Rentang tanggal train : 2023-03-03 00:00:00 s.d. 2024-12-31 00:00:00
Rentang tanggal val   : 2025-01-02 00:00:00 s.d. 2025-12-31 00:00:00
Rentang tanggal test  : 2026-01-02 00:00:00 s.d. 2026-02-28 00:00:00


## 6. Missing Value Handling Setelah Splitting

Missing value handling dilakukan setelah data dibagi menjadi train, validation, dan test agar statistik imputasi tidak menggunakan informasi dari masa depan. Pada tahap ini, fitur numerik diimputasi menggunakan median dari training set, sedangkan fitur kategorikal diisi dengan nilai `Unknown`. Pendekatan ini menjaga pipeline tetap bebas dari leakage dan membuat evaluasi model lebih realistis.

In [495]:
missing_train = (
    X_train.isna().sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)
missing_train["missing_pct"] = missing_train["missing_count"] / len(X_train) * 100

display(missing_train[missing_train["missing_count"] > 0].head(30))

,missing_count,missing_pct
cuisine_hist_max_score,90,0.227629
cuisine_hist_avg_score,90,0.227629
insp_type_hist_bad_rate,15,0.037938
boro_hist_critical_rate,6,0.015175


In [496]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()

print("Jumlah numeric cols    :", len(numeric_cols))
print("Jumlah categorical cols:", len(categorical_cols))

Jumlah numeric cols    : 42
Jumlah categorical cols: 9


In [497]:

from sklearn.impute import SimpleImputer

X_train_imp = X_train.copy()
X_val_imp = X_val.copy()
X_test_imp = X_test.copy()

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="constant", fill_value="Unknown")

if numeric_cols:
    X_train_imp[numeric_cols] = num_imputer.fit_transform(X_train[numeric_cols])
    X_val_imp[numeric_cols] = num_imputer.transform(X_val[numeric_cols])

if categorical_cols:
    X_train_imp[categorical_cols] = cat_imputer.fit_transform(X_train[categorical_cols])
    X_val_imp[categorical_cols] = cat_imputer.transform(X_val[categorical_cols])

print("Imputasi selesai.")
print("X_train_imp:", X_train_imp.shape)
print("X_val_imp  :", X_val_imp.shape)
print("X_test_imp :", X_test_imp.shape)

Imputasi selesai.
X_train_imp: (39538, 54)
X_val_imp  : (25590, 54)
X_test_imp : (3732, 54)


In [498]:
print("Missing di X_train_imp:", X_train_imp.isna().sum().sum())
print("Missing di X_val_imp  :", X_val_imp.isna().sum().sum())
print("Missing di X_test_imp :", X_test_imp.isna().sum().sum())

Missing di X_train_imp: 0
Missing di X_val_imp  : 0
Missing di X_test_imp : 0


**Penjelasan:** Missing value handling dilakukan setelah proses splitting agar nilai imputasi tidak dipengaruhi oleh data validation maupun test. Pada pipeline ini, fitur numerik diisi menggunakan median dari training set, sedangkan fitur kategorikal diisi dengan `Unknown`. Dengan pendekatan ini, preprocessing menjadi lebih konsisten dan evaluasi model tetap terjaga dari data leakage.

## 7. Audit Kardinalitas dan Penentuan Strategi Encoding

Pada tahap ini dilakukan pemeriksaan jumlah kategori unik pada setiap fitur kategorikal di training set. Tujuannya adalah menentukan strategi encoding yang paling sesuai untuk masing-masing kolom, misalnya one-hot encoding untuk kardinalitas rendah, frequency atau target encoding untuk kardinalitas tinggi, serta drop untuk fitur yang terlalu spesifik dan kurang generalizable. Audit dilakukan pada training set agar keputusan encoding tidak dipengaruhi oleh distribusi kategori pada validation dan test set.

In [499]:
cat_cols = X_train_imp.select_dtypes(include=["object", "category", "string"]).columns.tolist()

print("Jumlah kolom kategorikal:", len(cat_cols))
print(cat_cols)

Jumlah kolom kategorikal: 9
['CUISINE_DESCRIPTION', 'BORO', 'ZIPCODE', 'Community_Board', 'INSPECTION TYPE', 'DBA', 'ACTION', 'prev_grade', 'last_inspection_risk_bucket']


In [500]:
cardinality_df = pd.DataFrame({
    "column": cat_cols,
    "n_unique": [X_train_imp[col].nunique(dropna=False) for col in cat_cols],
    "missing_count": [X_train_imp[col].isna().sum() for col in cat_cols],
    "top_category": [X_train_imp[col].mode(dropna=False).iloc[0] if not X_train_imp[col].mode(dropna=False).empty else None for col in cat_cols],
    "top_freq": [X_train_imp[col].value_counts(dropna=False).iloc[0] if not X_train_imp[col].value_counts(dropna=False).empty else 0 for col in cat_cols]
})

cardinality_df["top_freq_pct"] = (cardinality_df["top_freq"] / len(X_train_imp) * 100).round(2)
cardinality_df = cardinality_df.sort_values(["n_unique", "top_freq_pct"], ascending=[True, False]).reset_index(drop=True)

display(cardinality_df)

,column,n_unique,missing_count,top_category,top_freq,top_freq_pct
0,last_inspection_risk_bucket,4,0,Unknown,32201,81.44
1,ACTION,5,0,Violations were cited in the following area(s).,37596,95.09
2,BORO,6,0,Manhattan,14897,37.68
3,prev_grade,7,0,Unknown,32069,81.11
4,INSPECTION TYPE,15,0,Cycle Inspection / Initial Inspection,22330,56.48
5,Community_Board,69,0,105.0,3357,8.49
6,CUISINE_DESCRIPTION,90,0,American,6611,16.72
7,ZIPCODE,222,0,10003.0,933,2.36
8,DBA,17711,0,DUNKIN,676,1.71


In [501]:
def cardinality_bucket(n):
    if n <= 10:
        return "low_cardinality"
    elif n <= 30:
        return "medium_cardinality"
    else:
        return "high_cardinality"

cardinality_df["cardinality_bucket"] = cardinality_df["n_unique"].apply(cardinality_bucket)
display(cardinality_df[["column", "n_unique", "cardinality_bucket", "top_category", "top_freq_pct"]])

,column,n_unique,cardinality_bucket,top_category,top_freq_pct
0,last_inspection_risk_bucket,4,low_cardinality,Unknown,81.44
1,ACTION,5,low_cardinality,Violations were cited in the following area(s).,95.09
2,BORO,6,low_cardinality,Manhattan,37.68
3,prev_grade,7,low_cardinality,Unknown,81.11
4,INSPECTION TYPE,15,medium_cardinality,Cycle Inspection / Initial Inspection,56.48
5,Community_Board,69,high_cardinality,105.0,8.49
6,CUISINE_DESCRIPTION,90,high_cardinality,American,16.72
7,ZIPCODE,222,high_cardinality,10003.0,2.36
8,DBA,17711,high_cardinality,DUNKIN,1.71


In [502]:
important_check_cols = [
    "BORO",
    "INSPECTION TYPE",
    "CUISINE_DESCRIPTION",
    "ZIPCODE",
    "DBA",
    "prev_grade",
    "last_inspection_risk_bucket"
]

for col in important_check_cols:
    if col in X_train_imp.columns:
        print(f"\n===== {col} =====")
        print("n_unique:", X_train_imp[col].nunique(dropna=False))
        display(X_train_imp[col].value_counts(dropna=False).head(15).to_frame("count"))


===== BORO =====
n_unique: 6


,count
BORO,
Manhattan,14897
Brooklyn,10463
Queens,8907
Bronx,3732
Staten Island,1520
Unknown,19



===== INSPECTION TYPE =====
n_unique: 15


,count
INSPECTION TYPE,
Cycle Inspection / Initial Inspection,22330
Cycle Inspection / Re-inspection,7851
Pre-permit (Operational) / Initial Inspection,5443
Pre-permit (Operational) / Re-inspection,1729
Cycle Inspection / Reopening Inspection,589
Pre-permit (Non-operational) / Initial Inspection,556
Inter-Agency Task Force / Initial Inspection,329
Pre-permit (Operational) / Compliance Inspection,239
Pre-permit (Operational) / Reopening Inspection,183



===== CUISINE_DESCRIPTION =====
n_unique: 90


...

,count
DBA,
DUNKIN,676
STARBUCKS,360
SUBWAY,330
MCDONALD'S,306
POPEYES,225
DUNKIN',184
BURGER KING,138
DOMINO'S,134
CHIPOTLE MEXICAN GRILL,133



===== prev_grade =====
n_unique: 7


,count
prev_grade,
Unknown,32069
A,6201
B,617
C,519
P,123
Z,5
N,4



===== last_inspection_risk_bucket =====
n_unique: 4


,count
last_inspection_risk_bucket,
Unknown,32201
low_risk,6201
medium_risk,617
high_risk,519


**Asumsi Awal**

In [503]:
one_hot_cols = [col for col in [
    "BORO",
    "INSPECTION TYPE",
    "prev_grade",
    "last_inspection_risk_bucket"
] if col in X_train_imp.columns]

freq_encode_cols = [col for col in [
    "CUISINE_DESCRIPTION",
    "ZIPCODE"
] if col in X_train_imp.columns]

drop_candidate_cols = [col for col in [
    "DBA",
    "ACTION"
] if col in X_train_imp.columns]

print("one_hot_cols:", one_hot_cols)
print("freq_encode_cols:", freq_encode_cols)
print("drop_candidate_cols:", drop_candidate_cols)

one_hot_cols: ['BORO', 'INSPECTION TYPE', 'prev_grade', 'last_inspection_risk_bucket']
freq_encode_cols: ['CUISINE_DESCRIPTION', 'ZIPCODE']
drop_candidate_cols: ['DBA', 'ACTION']


**Penjelasan:** Audit kardinalitas dilakukan untuk menentukan metode encoding yang paling sesuai bagi setiap fitur kategorikal. Fitur dengan jumlah kategori sedikit, seperti `BORO` dan `INSPECTION TYPE`, lebih cocok menggunakan one-hot encoding, sedangkan fitur dengan kategori lebih banyak, seperti `CUISINE DESCRIPTION` dan `ZIPCODE`, lebih stabil jika menggunakan frequency encoding atau target encoding berbasis cross-validation. Sementara itu, kolom seperti `DBA` dipertimbangkan untuk dibuang karena terlalu spesifik dan berisiko kurang generalizable.

## 8. Geo Clustering dari Latitude dan Longitude

Pada tahap ini dibuat fitur lokasi berbasis clustering menggunakan koordinat `Latitude` dan `Longitude`. Tujuannya adalah menangkap pola spasial yang mungkin tidak terlihat jika hanya menggunakan koordinat mentah, misalnya area-area dengan karakteristik risiko inspeksi yang mirip. Model KMeans di-fit hanya pada training set, lalu hasil cluster diterapkan ke validation dan test set untuk menjaga pipeline tetap bebas dari leakage.

In [504]:
X_train_fe = X_train_imp.copy()
X_val_fe = X_val_imp.copy()
X_test_fe = X_test_imp.copy()

lat_cols = ["Latitude"]
lon_cols = ["Longitude"]

lat_col = next((c for c in lat_cols if c in X_train_fe.columns), None)
lon_col = next((c for c in lon_cols if c in X_train_fe.columns), None)

print("Latitude col :", lat_col)
print("Longitude col:", lon_col)

Latitude col : Latitude
Longitude col: Longitude


In [505]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

geo_train = X_train_fe[[lat_col, lon_col]].copy()
geo_val = X_val_fe[[lat_col, lon_col]].copy()
geo_test = X_test_fe[[lat_col, lon_col]].copy()

geo_scaler = StandardScaler()
geo_train_scaled = geo_scaler.fit_transform(geo_train)
geo_val_scaled = geo_scaler.transform(geo_val)
geo_test_scaled = geo_scaler.transform(geo_test)

k_results = []

for k in range(3, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(geo_train_scaled)
    sil_score = silhouette_score(geo_train_scaled, labels, random_state=42)
    
    k_results.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette_score": sil_score
    })

k_eval_df = pd.DataFrame(k_results)
display(k_eval_df)

,k,inertia,silhouette_score
0,3,32798.812931,0.422967
1,4,21937.496332,0.475730
2,5,15648.911285,0.483302
3,6,13002.001597,0.464749
4,7,11220.657362,0.394950
5,8,9632.207331,0.412748
6,9,8570.264527,0.403805
7,10,7667.307793,0.403689


In [506]:
best_k = k_eval_df.sort_values("silhouette_score", ascending=False).iloc[0]["k"]
best_k = int(best_k)

print("Best k:", best_k)

geo_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
geo_kmeans.fit(geo_train_scaled)

Best k: 5


,"n_clusters n_clusters: int, default=8The number of clusters to form as well as the number ofcentroids to generate.For an example of how to choose an optimal value for `n_clusters` refer to:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_silhouette_analysis.py`.",5
,"init init: {'k-means++', 'random'}, callable or array-like of shape (n_clusters, n_features), default='k-means++'Method for initialization:* 'k-means++' : selects initial cluster centroids using sampling based on an empirical probability distribution of the points' contribution to the overall inertia. This technique speeds up convergence. The algorithm implemented is ""greedy k-means++"". It differs from the vanilla k-means++ by making several trials at each sampling step and choosing the best centroid among them.* 'random': choose `n_clusters` observations (rows) at random from data for the initial centroids.* If an array is passed, it should be of shape (n_clusters, n_features) and gives the initial centers.* If a callable is passed, it should take arguments X, n_clusters and a random state and return an initialization.For an example of how to use the different `init` strategies, see:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_digits.py`.For an evaluation of the impact of initialization, see the example:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_stability_low_dim_dense.py`.",'k-means++'
,"n_init n_init: 'auto' or int, default='auto'Number of times the k-means algorithm is run with different centroidseeds. The final results is the best output of `n_init` consecutive runsin terms of inertia. Several runs are recommended for sparsehigh-dimensional problems (see :ref:`kmeans_sparse_high_dim`).When `n_init='auto'`, the number of runs depends on the value of init:10 if using `init='random'` or `init` is a callable;1 if using `init='k-means++'` or `init` is an array-like... versionadded:: 1.2 Added 'auto' option for `n_init`... versionchanged:: 1.4 Default value for `n_init` changed to `'auto'`.",10
,"max_iter max_iter: int, default=300Maximum number of iterations of the k-means algorithm for asingle run.",300
,"tol tol: float, default=1e-4Relative tolerance with regards to Frobenius norm of the differencein the cluster centers of two consecutive iterations to declareconvergence.",0.0001
,"verbose verbose: int, default=0Verbosity mode.",0
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for centroid initialization. Usean int to make the randomness deterministic.See :term:`Glossary `.",42
,"copy_x copy_x: bool, default=TrueWhen pre-computing distances it is more numerically accurate to centerthe data first. If copy_x is True (default), then the original data isnot modified. If False, the original data is modified, and put backbefore the function returns, but small numerical differences may beintroduced by subtracting and then adding the data mean. Note that ifthe original data is not C-contiguous, a copy will be made even ifcopy_x is False. If the original data is sparse, but not in CSR format,a copy will be made even if copy_x is False.",True
,"algorithm algorithm: {""lloyd"", ""elkan""}, default=""lloyd""K-means algorithm to use. The classical EM-style algorithm is `""lloyd""`.The `""elkan""` variation can be more efficient on some datasets withwell-defined clusters, by using the triangle inequality. However it'smore memory intensive due to the allocation of an extra array of shape`(n_samples, n_clusters)`... versionchanged:: 0.18 Added Elkan algorithm.. versionchanged:: 1.1 Renamed ""full"" to ""lloyd"", and deprecated ""auto"" and ""full"". Changed ""auto"" to use ""lloyd"" instead of ""elkan"".",'lloyd'


In [507]:
X_train_fe["geo_cluster"] = geo_kmeans.predict(geo_train_scaled).astype("str")
X_val_fe["geo_cluster"] = geo_kmeans.predict(geo_val_scaled).astype("str")
X_test_fe["geo_cluster"] = geo_kmeans.predict(geo_test_scaled).astype("str")

print("Distribusi geo_cluster train:")
display(X_train_fe["geo_cluster"].value_counts().sort_index().to_frame("count"))

Distribusi geo_cluster train:


,count
geo_cluster,
0,1498
1,18205
2,6839
3,4887
4,8109


In [508]:
if "geo_cluster" in X_train_fe.columns and "geo_cluster" not in one_hot_cols:
    one_hot_cols.append("geo_cluster")

**Penjelasan:** Fitur lokasi tambahan dibuat menggunakan KMeans clustering pada koordinat `Latitude` dan `Longitude`. Pendekatan ini membantu model menangkap pola spasial dalam bentuk zona atau kelompok area, sehingga informasi lokasi tidak hanya direpresentasikan sebagai angka mentah. Selain label cluster, ditambahkan juga jarak ke centroid terdekat sebagai fitur numerik tambahan yang merepresentasikan kedekatan sebuah restoran terhadap pusat cluster lokasinya.

## 9. Pemeriksaan Skewness dan Log Transform

Pada tahap ini dilakukan pemeriksaan skewness pada fitur-fitur numerik tertentu untuk melihat apakah distribusinya sangat miring ke kanan. Jika suatu fitur terbukti right-skewed, maka diterapkan transformasi `log1p` agar distribusinya menjadi lebih stabil dan pengaruh nilai ekstrem dapat dikurangi. Pemeriksaan skewness dilakukan pada training set saja, lalu transformasi yang sama diterapkan ke validation dan test set.

In [509]:
skew_all = (
    X_train_fe.select_dtypes(include=["number"])
    .skew()
    .sort_values(ascending=False)
    .to_frame("skewness")
)

def skew_category(x):
    if x > 1:
        return "high right-skew"
    elif x > 0.5:
        return "moderate right-skew"
    elif x >= -0.5:
        return "roughly symmetric"
    elif x >= -1:
        return "moderate left-skew"
    else:
        return "high left-skew"

skew_all["category"] = skew_all["skewness"].apply(skew_category)

display(skew_all)

,skewness,category
prev_score_2,6.567535,high right-skew
ever_failed_gradeA_flag,4.330134,high right-skew
prev_score_1,3.143401,high right-skew
avg_prev_score,2.875837,high right-skew
repeat_offender_flag,2.800778,high right-skew
max_prev_score,2.672683,high right-skew
cuisine_hist_count,1.854053,high right-skew
inspection_count_so_far,1.805151,high right-skew
inspection_seq,1.805151,high right-skew
prev_critical_count,1.786630,high right-skew


In [510]:
X_train_tr = X_train_fe.copy()
X_val_tr = X_val_fe.copy()
X_test_tr = X_test_fe.copy()

candidate_log_cols = [
    "prev_score_1",
    "prev_score_2",
    "avg_prev_score",
    "max_prev_score",
    "prev_violation_count",
    "prev_critical_count",
    "avg_prev_critical_count",
    "days_since_last_inspection",
    "days_since_first_inspection",
    "inspection_count_so_far",
    "prior_critical_inspection_count",
    "cuisine_hist_count",
    "insp_type_hist_count",
]

candidate_log_cols = [col for col in candidate_log_cols if col in X_train_tr.columns]
print("Candidate log cols:", candidate_log_cols)

Candidate log cols: ['prev_score_1', 'prev_score_2', 'avg_prev_score', 'max_prev_score', 'prev_violation_count', 'prev_critical_count', 'avg_prev_critical_count', 'days_since_last_inspection', 'days_since_first_inspection', 'inspection_count_so_far', 'prior_critical_inspection_count', 'cuisine_hist_count', 'insp_type_hist_count']


In [511]:
skew_df = pd.DataFrame({
    "feature": candidate_log_cols,
    "skewness_train": [X_train_tr[col].skew() for col in candidate_log_cols],
    "min_train": [X_train_tr[col].min() for col in candidate_log_cols],
    "max_train": [X_train_tr[col].max() for col in candidate_log_cols]
}).sort_values("skewness_train", ascending=False).reset_index(drop=True)

display(skew_df)

,feature,skewness_train,min_train,max_train
0,prev_score_2,6.567535,0.0,153.0
1,prev_score_1,3.143401,0.0,153.0
2,avg_prev_score,2.875837,0.0,153.0
3,max_prev_score,2.672683,0.0,153.0
4,cuisine_hist_count,1.854053,0.0,6610.0
5,inspection_count_so_far,1.805151,0.0,8.0
6,prev_critical_count,1.786630,0.0,12.0
7,prior_critical_inspection_count,1.784247,0.0,7.0
8,avg_prev_critical_count,1.623514,0.0,12.0
9,prev_violation_count,1.509787,0.0,16.0


In [512]:
log_transform_cols = skew_df[
    (skew_df["skewness_train"] > 1) &
    (skew_df["min_train"] >= 0)
]["feature"].tolist()

print("Kolom yang akan di-log transform:")
print(log_transform_cols)

Kolom yang akan di-log transform:
['prev_score_2', 'prev_score_1', 'avg_prev_score', 'max_prev_score', 'cuisine_hist_count', 'inspection_count_so_far', 'prev_critical_count', 'prior_critical_inspection_count', 'avg_prev_critical_count', 'prev_violation_count', 'days_since_last_inspection', 'days_since_first_inspection']


In [513]:
for col in log_transform_cols:
    X_train_tr[col] = np.log1p(X_train_tr[col])
    X_val_tr[col] = np.log1p(X_val_tr[col])
    X_test_tr[col] = np.log1p(X_test_tr[col])

print("Log transform selesai.")

Log transform selesai.


In [514]:
if log_transform_cols:
    skew_after_df = pd.DataFrame({
        "feature": log_transform_cols,
        "skew_before": [X_train_fe[col].skew() for col in log_transform_cols],
        "skew_after": [X_train_tr[col].skew() for col in log_transform_cols],
        "min_before": [X_train_fe[col].min() for col in log_transform_cols],
        "max_before": [X_train_fe[col].max() for col in log_transform_cols],
        "min_after": [X_train_tr[col].min() for col in log_transform_cols],
        "max_after": [X_train_tr[col].max() for col in log_transform_cols]
    }).sort_values("skew_before", ascending=False).reset_index(drop=True)

    display(skew_after_df)
else:
    print("Tidak ada kolom yang memenuhi kriteria log transform.")

,feature,skew_before,skew_after,min_before,max_before,min_after,max_after
0,prev_score_2,6.567535,-2.032861,0.0,153.0,0.0,5.036953
1,prev_score_1,3.143401,-0.797492,0.0,153.0,0.0,5.036953
2,avg_prev_score,2.875837,-1.233970,0.0,153.0,0.0,5.036953
3,max_prev_score,2.672683,-1.418421,0.0,153.0,0.0,5.036953
4,cuisine_hist_count,1.854053,-0.826551,0.0,6610.0,0.0,8.796490
5,inspection_count_so_far,1.805151,0.759455,0.0,8.0,0.0,2.197225
6,prev_critical_count,1.786630,0.953303,0.0,12.0,0.0,2.564949
7,prior_critical_inspection_count,1.784247,0.851117,0.0,7.0,0.0,2.079442
8,avg_prev_critical_count,1.623514,0.856433,0.0,12.0,0.0,2.564949
9,prev_violation_count,1.509787,0.676961,0.0,16.0,0.0,2.833213


**Penjelasan:** Pada tahap ini diterapkan transformasi `log1p` pada fitur-fitur historis yang cenderung right-skewed, termasuk fitur skor historis, jumlah pelanggaran, serta fitur berbasis waktu dan jumlah inspeksi. Transformasi dilakukan untuk menstabilkan distribusi fitur, mengurangi pengaruh nilai ekstrem, dan membuat representasi numerik menjadi lebih halus bagi model. Daftar fitur yang ditransformasikan ditentukan dari hasil evaluasi distribusi pada training set, lalu transformasi yang sama diterapkan ke validation dan test set.

**Cek Kolom Score Natural Outlier atau bukan**

In [515]:
high_score_df = df_hist_clean[df_hist_clean["SCORE"] >= df_hist_clean["SCORE"].quantile(0.99)][
    ["CAMIS", "SCORE", "violation_count", "critical_count", "prev_score_1", "prev_critical_count", "INSPECTION TYPE", "BORO"]
].sort_values("SCORE", ascending=False)

display(
    high_score_df.groupby("CAMIS")["SCORE"]
    .agg(["count", "max", "mean"])
    .sort_values(["count", "max"], ascending=False)
    .head(30)
)

,count,max,mean
CAMIS,,,
50138270,4,138.0,104.500000
50048260,3,103.0,91.333333
50159074,3,95.0,90.333333
50130385,3,93.0,80.000000
50167110,2,125.0,113.500000
50144023,2,111.0,93.000000
50164558,2,109.0,105.000000
50150097,2,108.0,92.500000
50118198,2,107.0,97.500000


In [516]:
high_score_threshold = df_hist_clean["SCORE"].quantile(0.99)

suspect_rows = df_hist_clean[df_hist_clean["SCORE"] >= high_score_threshold].copy()

display(
    suspect_rows[
        [
            "CAMIS", "SCORE", "violation_count", "critical_count",
            "prev_score_1", "prev_critical_count",
            "INSPECTION TYPE", "BORO", "CUISINE_DESCRIPTION"
        ]
    ].sort_values("SCORE", ascending=False).head(30)
)

,CAMIS,SCORE,violation_count,critical_count,prev_score_1,prev_critical_count,INSPECTION TYPE,BORO,CUISINE_DESCRIPTION
68464,50177707,203.0,10,8,15.0,0.0,Pre-permit (Non-operational) / Initial Inspection,Manhattan,Other
68294,50176725,180.0,14,7,15.0,0.0,Pre-permit (Non-operational) / Initial Inspection,Brooklyn,Soul Food
49239,50127670,175.0,15,11,65.0,4.0,Cycle Inspection / Compliance Inspection,Manhattan,American
52436,50132754,168.0,9,8,15.0,0.0,Pre-permit (Operational) / Initial Inspection,Manhattan,Coffee/Tea
63645,50157149,168.0,14,10,15.0,0.0,Pre-permit (Operational) / Initial Inspection,Queens,Chinese
43508,50116369,154.0,11,8,19.0,3.0,Cycle Inspection / Re-inspection,Queens,Mexican
61413,50149744,153.0,7,6,15.0,0.0,Pre-permit (Non-operational) / Initial Inspection,Brooklyn,American
68036,50174896,144.0,9,6,15.0,0.0,Pre-permit (Non-operational) / Initial Inspection,Brooklyn,Korean
67437,50171473,144.0,7,5,15.0,0.0,Pre-permit (Operational) / Initial Inspection,Brooklyn,Other
38064,50105485,144.0,12,7,15.0,0.0,Pre-permit (Operational) / Initial Inspection,Brooklyn,Italian


In [517]:
id_counts_full = (
    df_hist_clean.groupby("CAMIS")["SCORE"]
    .agg(total_inspections="count", max_score="max", mean_score="mean")
    .sort_values("max_score", ascending=False)
)

display(id_counts_full.loc[[50177707, 50176725, 50127670]])

,total_inspections,max_score,mean_score
CAMIS,,,
50177707,1,203.0,203.000000
50176725,2,180.0,120.000000
50127670,7,175.0,49.714286


In [518]:
camis_to_check = [50177707, 50176725, 50127670] 

display(
    df_hist_clean[df_hist_clean["CAMIS"].isin(camis_to_check)]
    .sort_values(["CAMIS", "INSPECTION DATE"])
    [[
        "CAMIS", "INSPECTION DATE", "SCORE",
        "violation_count", "critical_count",
        "prev_score_1", "prev_critical_count",
        "INSPECTION TYPE"
    ]]
)

,CAMIS,INSPECTION DATE,SCORE,violation_count,critical_count,prev_score_1,prev_critical_count,INSPECTION TYPE
49235,50127670,2024-05-23,33.0,5,4,15.0,0.0,Cycle Inspection / Initial Inspection
49236,50127670,2024-12-19,30.0,5,3,33.0,4.0,Cycle Inspection / Re-inspection
49237,50127670,2025-03-25,24.0,4,3,30.0,3.0,Cycle Inspection / Initial Inspection
49238,50127670,2025-08-13,65.0,7,4,24.0,3.0,Cycle Inspection / Re-inspection
49239,50127670,2025-08-28,175.0,15,11,65.0,4.0,Cycle Inspection / Compliance Inspection
49240,50127670,2025-09-02,2.0,1,0,175.0,11.0,Cycle Inspection / Reopening Inspection
49241,50127670,2025-12-16,19.0,4,2,2.0,0.0,Cycle Inspection / Initial Inspection
68294,50176725,2025-10-09,180.0,14,7,15.0,0.0,Pre-permit (Non-operational) / Initial Inspection
68295,50176725,2026-01-05,60.0,4,2,180.0,7.0,Pre-permit (Non-operational) / Re-inspection
68464,50177707,2025-10-27,203.0,10,8,15.0,0.0,Pre-permit (Non-operational) / Initial Inspection


**Penjelasan:** Outlier yang ada tersebut kemungkinan besar merupakan natural outlier karena didukung oleh fitur-fitur lainnya seperti `violation_count`, `critical_count`, `Cuisine_Description`, `BORO`, dan `Inspection Type`.

## 10. Encoding Fitur Kategorikal

Pada tahap ini, fitur kategorikal diubah ke representasi numerik agar dapat digunakan oleh model machine learning. Fitur dengan kardinalitas rendah seperti `BORO`, `INSPECTION TYPE`, `prev_grade`, `last_inspection_risk_bucket`, dan `geo_cluster` diencoding menggunakan one-hot encoding, sedangkan fitur dengan kardinalitas lebih tinggi seperti `CUISINE DESCRIPTION` dan `ZIPCODE` direpresentasikan menggunakan frequency encoding. Kolom `DBA` tidak digunakan karena terlalu spesifik dan berisiko kurang generalizable.

In [519]:
from sklearn.preprocessing import OneHotEncoder

X_train_enc_in = X_train_tr.copy()
X_val_enc_in = X_val_tr.copy()
X_test_enc_in = X_test_tr.copy()

one_hot_cols = [col for col in [
    "BORO",
    "INSPECTION TYPE",
    "prev_grade",
    "last_inspection_risk_bucket",
    "geo_cluster"
] if col in X_train_enc_in.columns]

freq_encode_cols = [col for col in [
    "CUISINE_DESCRIPTION",
    "ZIPCODE"
] if col in X_train_enc_in.columns]

drop_candidate_cols = [col for col in [
    "DBA",
    "ACTION"
] if col in X_train_enc_in.columns]

print("one_hot_cols:", one_hot_cols)
print("freq_encode_cols:", freq_encode_cols)
print("drop_candidate_cols:", drop_candidate_cols)

one_hot_cols: ['BORO', 'INSPECTION TYPE', 'prev_grade', 'last_inspection_risk_bucket', 'geo_cluster']
freq_encode_cols: ['CUISINE_DESCRIPTION', 'ZIPCODE']
drop_candidate_cols: ['DBA', 'ACTION']


In [520]:
X_train_enc_in = X_train_enc_in.drop(columns=drop_candidate_cols, errors="ignore")
X_val_enc_in = X_val_enc_in.drop(columns=drop_candidate_cols, errors="ignore")
X_test_enc_in = X_test_enc_in.drop(columns=drop_candidate_cols, errors="ignore")

**Frequency Encoding**

In [521]:
freq_maps = {}

for col in freq_encode_cols:
    freq_map = X_train_enc_in[col].value_counts(normalize=True)
    freq_maps[col] = freq_map
    
    X_train_enc_in[f"{col}_freq"] = X_train_enc_in[col].map(freq_map)
    X_val_enc_in[f"{col}_freq"] = X_val_enc_in[col].map(freq_map)
    X_test_enc_in[f"{col}_freq"] = X_test_enc_in[col].map(freq_map)
    
    X_val_enc_in[f"{col}_freq"] = X_val_enc_in[f"{col}_freq"].fillna(0)
    X_test_enc_in[f"{col}_freq"] = X_test_enc_in[f"{col}_freq"].fillna(0)

**One Hot Encoding**

In [522]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

train_ohe = ohe.fit_transform(X_train_enc_in[one_hot_cols])
val_ohe = ohe.transform(X_val_enc_in[one_hot_cols])
test_ohe = ohe.transform(X_test_enc_in[one_hot_cols])

ohe_feature_names = ohe.get_feature_names_out(one_hot_cols)

X_train_ohe = pd.DataFrame(train_ohe, columns=ohe_feature_names, index=X_train_enc_in.index)
X_val_ohe = pd.DataFrame(val_ohe, columns=ohe_feature_names, index=X_val_enc_in.index)
X_test_ohe = pd.DataFrame(test_ohe, columns=ohe_feature_names, index=X_test_enc_in.index)

In [523]:
raw_cat_to_drop = one_hot_cols + freq_encode_cols

X_train_enc = pd.concat(
    [X_train_enc_in.drop(columns=raw_cat_to_drop, errors="ignore"), X_train_ohe],
    axis=1
)

X_val_enc = pd.concat(
    [X_val_enc_in.drop(columns=raw_cat_to_drop, errors="ignore"), X_val_ohe],
    axis=1
)

X_test_enc = pd.concat(
    [X_test_enc_in.drop(columns=raw_cat_to_drop, errors="ignore"), X_test_ohe],
    axis=1
)

print("X_train_enc:", X_train_enc.shape)
print("X_val_enc  :", X_val_enc.shape)
print("X_test_enc :", X_test_enc.shape)

X_train_enc: (39538, 85)
X_val_enc  : (25590, 85)
X_test_enc : (3732, 85)


In [524]:
print("Jumlah fitur akhir:", len(X_train_enc.columns))
print()
print("Contoh kolom hasil encoding:")
print(X_train_enc.columns.tolist()[:80])

Jumlah fitur akhir: 85

Contoh kolom hasil encoding:
['CAMIS', 'INSPECTION DATE', 'Latitude', 'Longitude', 'Community_Board', 'inspection_year', 'inspection_month', 'inspection_quarter', 'inspection_dayofweek', 'violation_count', 'critical_count', 'inspection_seq', 'is_first_inspection', 'has_history', 'prev_score_1', 'prev_score_2', 'avg_prev_score', 'max_prev_score', 'score_trend', 'score_delta_last', 'prev_violation_count', 'prev_critical_count', 'avg_prev_critical_count', 'prior_critical_inspection_count', 'repeat_offender_flag', 'prev_inspection_date', 'days_since_last_inspection', 'first_inspection_date', 'days_since_first_inspection', 'inspection_count_so_far', 'ever_failed_gradeA_flag', 'prev_score_1_missing_flag', 'prev_score_2_missing_flag', 'avg_prev_score_missing_flag', 'max_prev_score_missing_flag', 'days_since_last_inspection_missing_flag', 'prev_inspection_date_missing_flag', 'bad_inspection_flag', 'cuisine_hist_count', 'cuisine_hist_avg_score', 'cuisine_hist_max_score',

In [525]:
X_train_enc.head()

,CAMIS,INSPECTION DATE,Latitude,Longitude,Community_Board,inspection_year,inspection_month,inspection_quarter,inspection_dayofweek,violation_count,critical_count,inspection_seq,is_first_inspection,has_history,prev_score_1,prev_score_2,avg_prev_score,max_prev_score,score_trend,score_delta_last,prev_violation_count,prev_critical_count,avg_prev_critical_count,prior_critical_inspection_count,repeat_offender_flag,prev_inspection_date,days_since_last_inspection,first_inspection_date,days_since_first_inspection,inspection_count_so_far,ever_failed_gradeA_flag,prev_score_1_missing_flag,prev_score_2_missing_flag,avg_prev_score_missing_flag,max_prev_score_missing_flag,days_since_last_inspection_missing_flag,prev_inspection_date_missing_flag,bad_inspection_flag,cuisine_hist_count,cuisine_hist_avg_score,cuisine_hist_max_score,critical_violation_flag,boro_hist_count,boro_hist_critical_rate,insp_type_hist_count,insp_type_hist_bad_rate,CUISINE_DESCRIPTION_freq,ZIPCODE_freq,BORO_Bronx,BORO_Brooklyn,BORO_Manhattan,BORO_Queens,BORO_Staten Island,BORO_Unknown,INSPECTION TYPE_Cycle Inspection / Compliance Inspection,INSPECTION TYPE_Cycle Inspection / Initial Inspection,INSPECTION TYPE_Cycle Inspection / Re-inspection,INSPECTION TYPE_Cycle Inspection / Reopening Inspection,INSPECTION TYPE_Cycle Inspection / Second Compliance Inspection,INSPECTION TYPE_Inter-Agency Task Force / Initial Inspection,INSPECTION TYPE_Pre-permit (Non-operational) / Compliance Inspection,INSPECTION TYPE_Pre-permit (Non-operational) / Initial Inspection,INSPECTION TYPE_Pre-permit (Non-operational) / Re-inspection,INSPECTION TYPE_Pre-permit (Non-operational) / Second Compliance Inspection,INSPECTION TYPE_Pre-permit (Operational) / Compliance Inspection,INSPECTION TYPE_Pre-permit (Operational) / Initial Inspection,INSPECTION TYPE_Pre-permit (Operational) / Re-inspection,INSPECTION TYPE_Pre-permit (Operational) / Reopening Inspection,INSPECTION TYPE_Pre-permit (Operational) / Second Compliance Inspection,prev_grade_A,prev_grade_B,prev_grade_C,prev_grade_N,prev_grade_P,prev_grade_Unknown,prev_grade_Z,last_inspection_risk_bucket_Unknown,last_inspection_risk_bucket_high_risk,last_inspection_risk_bucket_low_risk,last_inspection_risk_bucket_medium_risk,geo_cluster_0,geo_cluster_1,geo_cluster_2,geo_cluster_3,geo_cluster_4
0,40385774.0,2023-03-03,40.697845,-73.991435,302.0,2023.0,3.0,1.0,4.0,10.0,5.0,1.0,1.0,0.0,2.772589,2.995732,2.981344,3.178054,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-03-03,0.0,2023-03-03,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.000000,17.082335,97.0,1.0,0.0,0.859109,0.0,0.371458,0.167206,0.017199,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,40397359.0,2023-03-03,40.764343,-73.988307,104.0,2023.0,3.0,1.0,4.0,5.0,2.0,1.0,1.0,0.0,2.772589,2.995732,2.981344,3.178054,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-03-03,0.0,2023-03-03,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.000000,17.082335,97.0,1.0,0.0,0.859109,0.0,0.371458,0.006854,0.022864,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,40423830.0,2023-03-03,40.887983,-73.860341,212.0,2023.0,3.0,1.0,4.0,3.0,2.0,1.0,1.0,0.0,2.772589,2.995732,2.981344,3.178054,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-03-03,0.0,2023-03-03,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.000000,17.082335,97.0,1.0,0.0,0.859109,0.0,0.371458,0.032298,0.003693,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,40534912.0,2023-03-03,40.686474,-73.974828,302.0,2023.0,3.0,1.0,4.0,5.0,2.0,1.0,1.0,0.0,2.772589,2.995732,2.981344,3.178054,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-03-03,0.0,2023-03-03,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.693147,11.000000,11.0,1.0,1.0,1.000000,1.0,1.000000,0.032298,0.010218,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,

**Penjelasan:** Encoding dilakukan setelah data dibagi agar semua transformasi statistik hanya belajar dari training set. Pada baseline ini, fitur kategorikal dengan kardinalitas rendah diencoding menggunakan one-hot encoding, sedangkan fitur dengan kardinalitas lebih tinggi menggunakan frequency encoding agar dimensi fitur tetap efisien. Pendekatan ini dipilih karena relatif stabil, mudah diinterpretasikan, dan lebih aman digunakan sebagai baseline dibanding langsung memakai target encoding.

## 13. Reduksi redundansi Fitur ##

In [526]:
X_train_sel = X_train_enc.copy()
X_val_sel = X_val_enc.copy()
X_test_sel = X_test_enc.copy()

**Berdasarkan Domain Knowledge**

In [527]:
manual_drop_cols = [
    "CAMIS",
    "violation_count",
    "critical_count",
    "bad_inspection_flag",
    "critical_violation_flag",
    "inspection_seq",
    "is_first_inspection",
    "has_history",
    "Latitude",
    "Longitude"
]

manual_drop_cols = [col for col in manual_drop_cols if col in X_train_sel.columns]

X_train_sel = X_train_sel.drop(columns=manual_drop_cols, errors="ignore")
X_val_sel = X_val_sel.drop(columns=manual_drop_cols, errors="ignore")
X_test_sel = X_test_sel.drop(columns=manual_drop_cols, errors="ignore")

print("Manual drop cols:")
print(manual_drop_cols)
print("Shape after manual drop:", X_train_sel.shape)

Manual drop cols:
['CAMIS', 'violation_count', 'critical_count', 'bad_inspection_flag', 'critical_violation_flag', 'inspection_seq', 'is_first_inspection', 'has_history', 'Latitude', 'Longitude']
Shape after manual drop: (39538, 75)


**Cek Fitur Konstanta**

In [528]:
near_constant_info = []

for col in X_train_sel.columns:
    top_freq_ratio = X_train_sel[col].value_counts(normalize=True, dropna=False).iloc[0]
    n_unique = X_train_sel[col].nunique(dropna=False)
    
    near_constant_info.append({
        "feature": col,
        "n_unique": n_unique,
        "top_freq_ratio": top_freq_ratio
    })

near_constant_df = pd.DataFrame(near_constant_info).sort_values(
    ["top_freq_ratio", "n_unique"], ascending=[False, True]
).reset_index(drop=True)

display(near_constant_df.head(30))

,feature,n_unique,top_freq_ratio
0,INSPECTION TYPE_Pre-permit (Non-operational) /...,2,0.999975
1,prev_grade_N,2,0.999899
2,prev_grade_Z,2,0.999874
3,INSPECTION TYPE_Pre-permit (Non-operational) /...,2,0.999823
4,INSPECTION TYPE_Cycle Inspection / Second Comp...,2,0.999671
5,BORO_Unknown,2,0.999519
6,INSPECTION TYPE_Pre-permit (Operational) / Sec...,2,0.999418
7,INSPECTION TYPE_Pre-permit (Non-operational) /...,2,0.997800
8,prev_grade_P,2,0.996889
9,INSPECTION TYPE_Cycle Inspection / Compliance ...,2,0.996004


In [529]:
near_constant_cols = near_constant_df.loc[
    (near_constant_df["n_unique"] == 1) | (near_constant_df["top_freq_ratio"] >= 0.995),
    "feature"
].tolist()

print("Near-constant cols:")
print(near_constant_cols)

Near-constant cols:
['INSPECTION TYPE_Pre-permit (Non-operational) / Second Compliance Inspection', 'prev_grade_N', 'prev_grade_Z', 'INSPECTION TYPE_Pre-permit (Non-operational) / Compliance Inspection', 'INSPECTION TYPE_Cycle Inspection / Second Compliance Inspection', 'BORO_Unknown', 'INSPECTION TYPE_Pre-permit (Operational) / Second Compliance Inspection', 'INSPECTION TYPE_Pre-permit (Non-operational) / Re-inspection', 'prev_grade_P', 'INSPECTION TYPE_Cycle Inspection / Compliance Inspection', 'INSPECTION TYPE_Pre-permit (Operational) / Reopening Inspection']


In [530]:
X_train_sel = X_train_sel.drop(columns=near_constant_cols, errors="ignore")
X_val_sel = X_val_sel.drop(columns=near_constant_cols, errors="ignore")
X_test_sel = X_test_sel.drop(columns=near_constant_cols, errors="ignore")

print("Shape after near-constant drop:", X_train_sel.shape)

Shape after near-constant drop: (39538, 64)


**Audit korelasi numerik tinggi**

In [531]:
corr_matrix = X_train_sel.select_dtypes(include=["number"]).corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "corr"})
    .sort_values("corr", ascending=False)
    .reset_index(drop=True)
)

high_corr_pairs = high_corr_pairs[high_corr_pairs["corr"] >= 0.95]

display(high_corr_pairs)

,feature_1,feature_2,corr
0,avg_prev_score_missing_flag,days_since_last_inspection_missing_flag,1.000000
1,avg_prev_score_missing_flag,prev_inspection_date_missing_flag,1.000000
2,prev_score_1_missing_flag,days_since_last_inspection_missing_flag,1.000000
3,days_since_last_inspection_missing_flag,prev_inspection_date_missing_flag,1.000000
4,max_prev_score_missing_flag,days_since_last_inspection_missing_flag,1.000000
5,max_prev_score_missing_flag,prev_inspection_date_missing_flag,1.000000
6,prev_score_1_missing_flag,avg_prev_score_missing_flag,1.000000
7,prev_grade_B,last_inspection_risk_bucket_medium_risk,1.000000
8,prev_score_1_missing_flag,max_prev_score_missing_flag,1.000000
9,prev_grade_C,last_inspection_risk_bucket_high_risk,1.000000


**Pasangan yang sangat mirip**

In [532]:
redundant_drop_cols = [
    "last_inspection_risk_bucket_high_risk",
    "last_inspection_risk_bucket_medium_risk",
    "last_inspection_risk_bucket_low_risk",
    "last_inspection_risk_bucket_Unknown",

    "prev_inspection_date_missing_flag",
    "max_prev_score_missing_flag",
    "avg_prev_score_missing_flag",
    "prev_score_1_missing_flag",

    "days_since_first_inspection",
    "avg_prev_critical_count",
    "inspection_quarter",
    "max_prev_score",
    "days_since_last_inspection_missing_flag",
    "geo_cluster_0"
]

In [533]:
redundant_drop_cols = [col for col in redundant_drop_cols if col in X_train_sel.columns]

X_train_sel = X_train_sel.drop(columns=redundant_drop_cols, errors="ignore")
X_val_sel = X_val_sel.drop(columns=redundant_drop_cols, errors="ignore")
X_test_sel = X_test_sel.drop(columns=redundant_drop_cols, errors="ignore")

print("Redundant drop cols:")
print(redundant_drop_cols)
print("Shape after redundant drop:", X_train_sel.shape)

Redundant drop cols:
['last_inspection_risk_bucket_high_risk', 'last_inspection_risk_bucket_medium_risk', 'last_inspection_risk_bucket_low_risk', 'last_inspection_risk_bucket_Unknown', 'prev_inspection_date_missing_flag', 'max_prev_score_missing_flag', 'avg_prev_score_missing_flag', 'prev_score_1_missing_flag', 'days_since_first_inspection', 'avg_prev_critical_count', 'inspection_quarter', 'max_prev_score', 'days_since_last_inspection_missing_flag', 'geo_cluster_0']
Shape after redundant drop: (39538, 50)


In [534]:
corr_matrix = X_train_sel.select_dtypes(include=["number"]).corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "corr"})
    .sort_values("corr", ascending=False)
    .reset_index(drop=True)
)

high_corr_pairs = high_corr_pairs[high_corr_pairs["corr"] >= 0.95]

display(high_corr_pairs)

,feature_1,feature_2,corr
0,prev_violation_count,prev_critical_count,0.952598
1,prior_critical_inspection_count,inspection_count_so_far,0.951556


**Penjelasan:** Reduksi redundansi fitur dilakukan untuk menyederhanakan representasi data sebelum modeling. Pada tahap ini dibuang fitur yang bersifat identitas, berpotensi menyebabkan leakage, memiliki variasi yang sangat rendah, atau terlalu mirip dengan fitur lain. Dengan demikian, himpunan fitur akhir menjadi lebih ringkas, lebih stabil, dan lebih relevan untuk digunakan pada tahap modeling berikutnya.

## 14. Scaling Fitur Numerik

Pada tahap ini dilakukan scaling pada fitur numerik kontinu menggunakan `RobustScaler`. Metode ini dipilih karena lebih tahan terhadap nilai ekstrem dibandingkan scaler berbasis mean dan standard deviation. Scaler di-fit hanya pada training set, lalu transformasi yang sama diterapkan ke validation dan test set agar pipeline tetap bebas dari leakage.

In [535]:
from sklearn.preprocessing import RobustScaler

X_train_final = X_train_sel.copy()
X_val_final = X_val_sel.copy()
X_test_final = X_test_sel.copy()

In [536]:
scaled_after_log_cols = [
    "prev_score_1",
    "prev_score_2",
    "avg_prev_score",
    "max_prev_score",
    "prev_violation_count",
    "prev_critical_count",
    "avg_prev_critical_count",
    "days_since_last_inspection",
    "days_since_first_inspection",
    "inspection_count_so_far",
    "prior_critical_inspection_count",
    "cuisine_hist_count",
    "insp_type_hist_count",
]

robust_only_cols = [
    "score_trend",
    "score_delta_last",
    "cuisine_hist_avg_score",
    "cuisine_hist_max_score",
    "boro_hist_critical_rate",
    "insp_type_hist_bad_rate",
]

scale_cols = [
    col for col in scaled_after_log_cols + robust_only_cols
    if col in X_train_final.columns
]

print("Kolom yang akan discale:")
print(scale_cols)
print("Jumlah:", len(scale_cols))

Kolom yang akan discale:
['prev_score_1', 'prev_score_2', 'avg_prev_score', 'prev_violation_count', 'prev_critical_count', 'days_since_last_inspection', 'inspection_count_so_far', 'prior_critical_inspection_count', 'cuisine_hist_count', 'insp_type_hist_count', 'score_trend', 'score_delta_last', 'cuisine_hist_avg_score', 'cuisine_hist_max_score', 'boro_hist_critical_rate', 'insp_type_hist_bad_rate']
Jumlah: 16


In [537]:
robust_scaler = RobustScaler()

if scale_cols:
    X_train_final[scale_cols] = robust_scaler.fit_transform(X_train_final[scale_cols])
    X_val_final[scale_cols] = robust_scaler.transform(X_val_final[scale_cols])
    X_test_final[scale_cols] = robust_scaler.transform(X_test_final[scale_cols])

print("Scaling selesai.")
print("X_train_final:", X_train_final.shape)
print("X_val_final  :", X_val_final.shape)
print("X_test_final :", X_test_final.shape)
display(X_train_final[scale_cols].head())

Scaling selesai.
X_train_final: (39538, 50)
X_val_final  : (25590, 50)
X_test_final : (3732, 50)


,prev_score_1,prev_score_2,avg_prev_score,prev_violation_count,prev_critical_count,days_since_last_inspection,inspection_count_so_far,prior_critical_inspection_count,cuisine_hist_count,insp_type_hist_count,score_trend,score_delta_last,cuisine_hist_avg_score,cuisine_hist_max_score,boro_hist_critical_rate,insp_type_hist_bad_rate
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.276035,-0.505464,0.0,0.0,0.000000,0.000000,0.000000,0.000000
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.276035,-0.505464,0.0,0.0,0.000000,0.000000,0.000000,0.000000
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.276035,-0.505464,0.0,0.0,0.000000,0.000000,0.000000,0.000000
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.923743,-0.505368,0.0,0.0,-1.391927,-1.622642,5.775798,14.122903
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.923743,-0.505368,0.0,0.0,6.388889,-0.981132,5.775798,14.122903


**Penjelasan:** Scaling dilakukan pada fitur numerik kontinu menggunakan `RobustScaler` agar perbedaan skala antar fitur menjadi lebih seimbang tanpa terlalu sensitif terhadap nilai ekstrem. Fitur yang telah ditransformasi log maupun fitur numerik kontinu lainnya di-scale menggunakan parameter dari training set saja, lalu transformasi yang sama diterapkan ke validation dan test set. Sementara itu, fitur kategorikal hasil one-hot encoding dan fitur biner tidak di-scale karena sudah berada dalam representasi numerik yang stabil.